# Notebook 09: Comprehensive Strategy Universe and Interactive Explorer

This notebook is a strategy atlas. It enumerates the historically defined FX carry strategy design space from Notebooks 06, 07, and 08, reconciles available canonical strategy identities, and builds common comparison, cost, risk-period, drawdown-protection, and interactive exploration outputs.

It is not a new optimization exercise. Rankings are descriptive views of historical specifications; the maximum Sharpe ratio in this notebook is not a newly validated trading rule.

## 0. Objective and Scope

The notebook uses the latest saved research artifacts rather than refetching Bloomberg data or modifying previous notebooks. It distinguishes:

- configuration rows: every historically named specification or alias;
- canonical rows: unique implemented weight streams;
- primary, robustness, exploratory, sensitivity, excluded, superseded, and duplicate specifications;
- own-signal episodes, broad rule-defined regimes, and fixed historical crises.

The performance framework follows the conventions inherited from Notebooks 07 and 08.

In [1]:
from pathlib import Path
import json
import hashlib
import math
import warnings

warnings.filterwarnings("ignore", category=UserWarning, message=".*Tight layout.*")
warnings.filterwarnings("ignore", category=DeprecationWarning, message=".*DataFrameGroupBy.apply operated on the grouping columns.*")

import numpy as np
import pandas as pd
import nbformat
from IPython.display import display, Markdown

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    sns = None

ROOT = Path.cwd()
if not (ROOT / "theo").exists():
    candidates = [p for p in Path.cwd().parents if (p / "theo").exists()]
    if not candidates:
        raise RuntimeError("Could not locate project root containing theo/.")
    ROOT = candidates[0]

PROCESSED = ROOT / "theo/data/processed"
OUT_DIR = PROCESSED / "09_strategy_universe"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

NOTEBOOK_PATH = ROOT / "theo/09_comprehensive_strategy_universe_and_interactive_explorer.ipynb"

SOURCE_NOTEBOOKS = [
    ("historical_option_source", "06_options_skew_and_vol_filters.ipynb", "historical option-filter concepts and legacy families"),
    ("authoritative_option_regression_source", "06_options_filter_regression(9).ipynb", "authoritative option-signal and regression definitions"),
    ("authoritative_option_strategy_source", "07_option_conditioned_carry_strategy.ipynb", "authoritative implementable option strategy definitions"),
    ("latest_integrated_source", "08_comprehensive_carry_strategy_optimization(6).ipynb", "latest macro, PIT, classification, and integrated strategy source"),
]

OUTPUTS = {
    "source_audit": OUT_DIR / "09_source_audit.parquet",
    "configuration_registry": OUT_DIR / "09_configuration_registry.parquet",
    "canonical_strategy_registry": OUT_DIR / "09_canonical_strategy_registry.parquet",
    "configuration_alias_audit": OUT_DIR / "09_configuration_alias_audit.parquet",
    "strategy_universe_coverage_audit": OUT_DIR / "09_strategy_universe_coverage_audit.parquet",
    "all_configuration_results": OUT_DIR / "09_all_configuration_results.parquet",
    "canonical_strategy_leaderboard": OUT_DIR / "09_canonical_strategy_leaderboard.parquet",
    "sample_performance": OUT_DIR / "09_sample_performance.parquet",
    "abcd_factorial_registry": OUT_DIR / "09_abcd_factorial_registry.parquet",
    "abcd_factorial_performance": OUT_DIR / "09_abcd_factorial_performance.parquet",
    "abcd_factorial_effects": OUT_DIR / "09_abcd_factorial_effects.parquet",
    "abcd_factorial_interactions": OUT_DIR / "09_abcd_factorial_interactions.parquet",
    "macro_overlay_results": OUT_DIR / "09_macro_overlay_results.parquet",
    "macro_composite_results": OUT_DIR / "09_macro_composite_results.parquet",
    "option_macro_crossproduct": OUT_DIR / "09_option_macro_crossproduct.parquet",
    "cost_sensitivity": OUT_DIR / "09_cost_sensitivity.parquet",
    "risk_episode_detail": OUT_DIR / "09_risk_episode_detail.parquet",
    "event_drawdown_zscores": OUT_DIR / "09_event_drawdown_zscores.parquet",
    "risk_signal_summary": OUT_DIR / "09_risk_signal_summary.parquet",
    "risk_family_summary": OUT_DIR / "09_risk_family_summary.parquet",
    "fixed_crisis_summary": OUT_DIR / "09_fixed_crisis_summary.parquet",
    "overall_risk_dd_summary": OUT_DIR / "09_overall_risk_dd_summary.parquet",
    "notebook08_reconciliation_audit": OUT_DIR / "09_notebook08_reconciliation_audit.parquet",
    "notebook07_reconciliation_audit": OUT_DIR / "09_notebook07_reconciliation_audit.parquet",
    "interactive_test_audit": OUT_DIR / "09_interactive_test_audit.parquet",
    "integrity_audit": OUT_DIR / "09_integrity_audit.parquet",
    "excel_workbook": OUT_DIR / "09_strategy_universe_results.xlsx",
    "summary": OUT_DIR / "09_strategy_universe_summary.json",
}

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def stable_hash_frame(df, cols):
    if df is None or df.empty:
        return ""
    d = df[list(cols)].copy()
    for c in d.columns:
        if np.issubdtype(d[c].dtype, np.datetime64):
            d[c] = pd.to_datetime(d[c]).dt.strftime("%Y-%m-%d")
    d = d.sort_values(list(cols)).reset_index(drop=True)
    payload = d.to_csv(index=False, float_format="%.17g").encode()
    return hashlib.sha256(payload).hexdigest()

def short_hash(text, n=20):
    return hashlib.sha256(str(text).encode()).hexdigest()[:n]

def readp(name, required=True):
    path = PROCESSED / name
    if required and not path.exists():
        raise FileNotFoundError(f"Required processed artifact missing: {path}")
    if not path.exists():
        return pd.DataFrame()
    return pd.read_parquet(path)

def max_drawdown(returns):
    r = pd.Series(returns).dropna()
    wealth = pd.concat([pd.Series([1.0]), (1.0 + r).cumprod()], ignore_index=True)
    dd = wealth / wealth.cummax() - 1.0
    return float(dd.min()) if len(dd) else np.nan

def expected_shortfall(returns, q=0.05):
    r = pd.Series(returns).dropna()
    if r.empty:
        return np.nan
    cutoff = r.quantile(q)
    tail = r[r <= cutoff]
    return float(tail.mean()) if not tail.empty else np.nan

def perf_stats(df, ret_col="gross_return"):
    if df is None or df.empty or ret_col not in df:
        return {
            "months": 0, "annualized_arithmetic_return": np.nan, "annualized_volatility": np.nan,
            "gross_sharpe": np.nan, "sortino_ratio": np.nan, "maximum_drawdown": np.nan,
            "expected_shortfall_5pct": np.nan, "hit_rate": np.nan, "skewness": np.nan,
            "worst_month": pd.NaT, "best_month": pd.NaT, "final_nav": np.nan,
            "average_long_gross": np.nan, "average_short_gross": np.nan, "average_total_gross": np.nan,
            "average_net_exposure": np.nan, "average_one_way_turnover": np.nan,
            "long_leg_annualized_contribution": np.nan, "short_leg_annualized_contribution": np.nan,
        }
    d = df.dropna(subset=[ret_col]).sort_values("month_end").copy()
    r = d[ret_col].astype(float)
    vol = float(r.std(ddof=1) * np.sqrt(12)) if len(r) > 1 else np.nan
    ann = float(r.mean() * 12) if len(r) else np.nan
    downside = r[r < 0].std(ddof=1) * np.sqrt(12) if (r < 0).sum() > 1 else np.nan
    return {
        "months": int(len(r)),
        "annualized_arithmetic_return": ann,
        "annualized_volatility": vol,
        "gross_sharpe": ann / vol if pd.notna(vol) and vol != 0 else np.nan,
        "sortino_ratio": ann / downside if pd.notna(downside) and downside != 0 else np.nan,
        "maximum_drawdown": max_drawdown(r),
        "expected_shortfall_5pct": expected_shortfall(r),
        "hit_rate": float((r > 0).mean()) if len(r) else np.nan,
        "skewness": float(r.skew()) if len(r) > 2 else np.nan,
        "worst_month": d.loc[r.idxmin(), "month_end"] if len(r) else pd.NaT,
        "best_month": d.loc[r.idxmax(), "month_end"] if len(r) else pd.NaT,
        "final_nav": float((1.0 + r).prod()) if len(r) else np.nan,
        "average_long_gross": float(d["long_gross"].mean()) if "long_gross" in d else np.nan,
        "average_short_gross": float(d["short_gross"].mean()) if "short_gross" in d else np.nan,
        "average_total_gross": float(d["gross_roll_notional"].mean()) if "gross_roll_notional" in d else float(d["total_gross"].mean()) if "total_gross" in d else np.nan,
        "average_net_exposure": float(d["net_exposure"].mean()) if "net_exposure" in d else np.nan,
        "average_one_way_turnover": float(d["one_way_turnover"].mean()) if "one_way_turnover" in d else np.nan,
        "long_leg_annualized_contribution": float(d["em_return_contribution"].fillna(0).mean() * 12) if "em_return_contribution" in d else np.nan,
        "short_leg_annualized_contribution": float(d["g10_return_contribution"].fillna(0).mean() * 12) if "g10_return_contribution" in d else np.nan,
    }

def display_df(title, df, n=20, fmt=None):
    display(Markdown(f"**{title}**"))
    if df is None or df.empty:
        display(Markdown("_No rows available._"))
        return
    sty = df.head(n).style
    if fmt:
        sty = sty.format(fmt, na_rep="")
    display(sty)

figure_paths = []
def savefig(fig, name):
    path = FIG_DIR / name
    fig.savefig(path, dpi=150, bbox_inches="tight")
    figure_paths.append(str(path))
    plt.close(fig)
    return path

display(Markdown(f"Output directory: `{OUT_DIR}`"))

Output directory: `/Users/theoli/Documents/Work/UChicago/Courses/36000_project_lab/BofA/repo/theo/data/processed/09_strategy_universe`

## 1. Source Audit

The four source notebooks are resolved by exact authoritative filenames. If a required source is missing, the notebook stops before producing research outputs.

In [2]:
audit_rows = []
for source_role, requested_filename, notes in SOURCE_NOTEBOOKS:
    candidates = sorted((ROOT / "theo").rglob(requested_filename))
    selected = ROOT / "theo" / requested_filename
    if len(candidates) > 1:
        print(f"Multiple candidates for {requested_filename}:")
        for c in candidates:
            print(" -", c)
        print("Selected exact authoritative project path:", selected)
    exists = selected.exists()
    if not exists:
        raise FileNotFoundError(f"Required source notebook missing: {selected}")
    nb_src = nbformat.read(selected, as_version=4)
    audit_rows.append({
        "source_role": source_role,
        "requested_filename": requested_filename,
        "resolved_path": str(selected),
        "exists": exists,
        "file_size": selected.stat().st_size,
        "sha256": sha256_file(selected),
        "notebook_cell_count": len(nb_src.cells),
        "code_cell_count": sum(c.cell_type == "code" for c in nb_src.cells),
        "markdown_cell_count": sum(c.cell_type == "markdown" for c in nb_src.cells),
        "selected": True,
        "notes": notes,
    })
source_audit = pd.DataFrame(audit_rows)
source_audit.to_parquet(OUTPUTS["source_audit"], index=False)
display_df("Source audit", source_audit, n=10)
assert source_audit["exists"].all()

Multiple candidates for 08_comprehensive_carry_strategy_optimization(6).ipynb:
 - /Users/theoli/Documents/Work/UChicago/Courses/36000_project_lab/BofA/repo/theo/08_comprehensive_carry_strategy_optimization(6).ipynb
 - /Users/theoli/Documents/Work/UChicago/Courses/36000_project_lab/BofA/repo/theo/old_version/08_comprehensive_carry_strategy_optimization(6).ipynb
Selected exact authoritative project path: /Users/theoli/Documents/Work/UChicago/Courses/36000_project_lab/BofA/repo/theo/08_comprehensive_carry_strategy_optimization(6).ipynb


**Source audit**

,source_role,requested_filename,resolved_path,exists,file_size,sha256,notebook_cell_count,code_cell_count,markdown_cell_count,selected,notes
0,historical_option_source,06_options_skew_and_vol_filters.ipynb,/Users/theoli/Documents/Work/UChicago/Courses/36000_project_lab/BofA/repo/theo/06_options_skew_and_vol_filters.ipynb,True,7210127,21fafdec2887f3567ac6a98fe7bc42efa2fab8104c17eb603673665df2e2cf8d,65,34,31,True,historical option-filter concepts and legacy families
1,authoritative_option_regression_source,06_options_filter_regression(9).ipynb,/Users/theoli/Documents/Work/UChicago/Courses/36000_project_lab/BofA/repo/theo/06_options_filter_regression(9).ipynb,True,8828619,182497cd35788b050dd8da7f092699d487b7a144c0586ca900574c9a8c0f7ed8,40,20,20,True,authoritative option-signal and regression definitions
2,authoritative_option_strategy_source,07_option_conditioned_carry_strategy.ipynb,/Users/theoli/Documents/Work/UChicago/Courses/36000_project_lab/BofA/repo/theo/07_option_conditioned_carry_strategy.ipynb,True,3448699,35ac663859ee63101cae7f3033e51886b3a5c7a08a1545572d82467dd39a5bf1,49,24,25,True,authoritative implementable option strategy definitions
3,latest_integrated_source,08_comprehensive_carry_strategy_optimization(6).ipynb,/Users/theoli/Documents/Work/UChicago/Courses/36000_project_lab/BofA/repo/theo/08_comprehensive_carry_strategy_optimization(6).ipynb,True,22766722,ba160c9571684848ac5ec73a15d9b4b80d05c8c08799e5a258c978ca1e2592a5,24,12,12,True,"latest macro, PIT, classification, and integrated strategy source"


## 2. Historical Strategy Taxonomy and Artifact Loading

The atlas loads saved outputs from prior notebooks. Notebook 08 is the authoritative integrated source for canonical strategy identity, PIT status, macro definitions, classification, cost analysis, and risk-period analysis. Notebook 07 provides detailed option-conditioned monthly return and weight streams where persisted.

In [3]:
# Notebook 08 authoritative artifacts
reg08 = readp("08v6_canonical_strategy_registry.parquet")
alias08 = readp("08v6_strategy_alias_registry.parquet")
class08 = readp("08v6_full_strategy_classification_registry.parquet")
decision08 = readp("08v6_cross_family_decision_table.parquet")
eval08 = readp("08v6_evaluation_leaderboard.parquet")
cost08 = readp("08v6_cost_sensitivity.parquet")
macro_features = readp("08v6_macro_feature_audit.parquet")
macro_events = readp("08v6_macro_event_audit.parquet")
macro_evidence = readp("08v6_predictive_evidence_map.parquet")
macro_composites = readp("08v6_composite_macro_comparison.parquet")
macro_ext = readp("08v6_macro_extension_incremental_results.parquet")
ultimate = readp("08v6_ultimate_exploratory_matrix.parquet")
risk_period_results = readp("08v6_risk_period_results.parquet")
risk_episode_source = readp("08v6_risk_period_episode_audit.parquet")
fixed_crisis_source = readp("08v6_strategy_fixed_crisis_performance.parquet")
pit08 = readp("08v6_strategy_no_lookahead_audit.parquet")
stats08 = readp("08v6_gross_statistical_tests.parquet")
summary08_path = PROCESSED / "08v6_summary.json"
summary08 = json.loads(summary08_path.read_text()) if summary08_path.exists() else {}

# Notebook 07 and historical option artifacts
option_returns = readp("option_conditioned_carry_returns.parquet", required=False)
option_weights = readp("option_conditioned_carry_weights.parquet", required=False)
option_perf = readp("option_conditioned_carry_performance.parquet", required=False)
option_grid = readp("option_conditioned_carry_full_strategy_grid.parquet", required=False)
legacy07_returns = readp("07a_conditioned_carry_returns.parquet", required=False)
legacy07_weights = readp("07a_conditioned_carry_weights.parquet", required=False)
legacy07_perf = readp("07a_conditioned_carry_performance.parquet", required=False)
old_filter_perf = readp("option_filter_performance_revised.parquet", required=False)
old_filter_returns = readp("option_filtered_carry_returns_revised.parquet", required=False)
long_side_perf = readp("option_long_side_strategy_performance.parquet", required=False)
long_side_returns = readp("option_long_side_strategy_returns.parquet", required=False)
long_side_weights = readp("option_long_side_strategy_weights.parquet", required=False)

loaded_artifacts = pd.DataFrame([
    {"artifact": name, "rows": len(obj), "columns": len(obj.columns) if isinstance(obj, pd.DataFrame) else 0}
    for name, obj in [
        ("08 canonical registry", reg08), ("08 alias registry", alias08), ("08 classification", class08),
        ("08 decision table", decision08), ("08 evaluation", eval08), ("08 cost", cost08),
        ("08 macro features", macro_features), ("08 macro events", macro_events),
        ("08 risk periods", risk_period_results), ("08 risk episodes", risk_episode_source),
        ("07 option returns", option_returns), ("07 option weights", option_weights),
        ("07a legacy returns", legacy07_returns), ("old filter returns", old_filter_returns),
        ("long-side option returns", long_side_returns),
    ]
])
display_df("Loaded artifacts", loaded_artifacts, n=30)
assert len(reg08) == 271, f"Expected Notebook 08 canonical registry to have 271 rows; got {len(reg08)}"

**Loaded artifacts**

,artifact,rows,columns
0,08 canonical registry,271,27
1,08 alias registry,323,6
2,08 classification,271,34
3,08 decision table,271,34
4,08 evaluation,271,17
5,08 cost,271,21
6,08 macro features,25,17
7,08 macro events,25,9
8,08 risk periods,17615,30
9,08 risk episodes,5735,12


## 3. Configuration Registry

The configuration registry keeps every historically named specification visible, including duplicate aliases. The canonical registry collapses identical implemented weight streams.

In [4]:
def status_from_role(row):
    tier = str(row.get("classification_tier", ""))
    role = str(row.get("evaluation_role", ""))
    label = str(row.get("label", ""))
    if tier == "Reference / not ranked" or label == "reference benchmark":
        return "reference"
    if role == "primary frozen strategy":
        return "primary frozen"
    if role == "core robustness strategy":
        return "core robustness"
    if role == "secondary robustness strategy":
        return "secondary robustness"
    if role == "pre-specified candidate family":
        return "pre-specified candidate"
    if role == "sensitivity strategy":
        return "sensitivity"
    if "exploratory" in role:
        return "exploratory"
    if "excluded" in label:
        return "excluded from primary analysis"
    return "secondary robustness"

class_lookup = class08.set_index("strategy").to_dict("index") if not class08.empty else {}
reg_lookup = reg08.set_index("strategy").to_dict("index")
alias_to_cid = alias08.set_index("alias")["canonical_strategy_id"].to_dict() if not alias08.empty else {}
cid_to_reg = reg08.set_index("canonical_strategy_id").to_dict("index")

config_rows = []
for _, a in alias08.iterrows():
    cid = a["canonical_strategy_id"]
    canon = cid_to_reg.get(cid, {})
    alias = str(a["alias"])
    cls = class_lookup.get(str(canon.get("strategy", alias)), class_lookup.get(alias, {}))
    hist = status_from_role(cls)
    config_rows.append({
        "configuration_id": short_hash("08-alias|" + alias + "|" + str(cid)),
        "display_name": alias,
        "source_notebook": "08_comprehensive_carry_strategy_optimization(6).ipynb",
        "source_section": "canonical registry / alias registry",
        "strategy_family": canon.get("strategy_family", ""),
        "strategy_subfamily": canon.get("option_architecture", ""),
        "selection_components": canon.get("option_architecture", ""),
        "risk_control_components": canon.get("macro_rule", ""),
        "macro_components": canon.get("strategy_macro_components", canon.get("macro_component", "")),
        "leg_allocation_components": canon.get("long_short_tilt", ""),
        "canonical_components": canon.get("option_architecture", ""),
        "canonical_parameters": canon.get("parameter_signature", ""),
        "stage_order": canon.get("stage_sequence", ""),
        "parameter_summary": canon.get("parameter_signature", ""),
        "historical_research_status": hist,
        "historically_primary": hist in {"reference", "primary frozen"},
        "historically_exploratory": hist == "exploratory",
        "historically_excluded": bool(a.get("excluded_from_ranking", False)) or hist == "excluded from primary analysis",
        "historically_sensitivity": hist == "sensitivity",
        "historically_superseded": False,
        "exclusion_reason": "excluded duplicate alias from ranking" if bool(a.get("excluded_from_ranking", False)) else "",
        "exclusion_reason_source": "08 alias registry" if bool(a.get("excluded_from_ranking", False)) else "",
        "regression_evidence_status": cls.get("predictive_evidence_status", ""),
        "constructible": True,
        "construction_status": "implemented canonical Notebook 08 weight stream",
        "nonconstructible_reason": "",
        "is_duplicate_alias": bool(a.get("excluded_from_ranking", False)),
        "duplicate_of_canonical_strategy_id": cid if bool(a.get("excluded_from_ranking", False)) else "",
        "canonical_strategy_id": cid,
        "notes": "historical alias retained even when duplicate",
    })

def add_historical_configs(df, source_notebook, source_section, name_col="strategy"):
    if df is None or df.empty or name_col not in df:
        return
    cols = [name_col]
    for c in ["strategy_family", "primary_or_exploratory", "parameter_values"]:
        if c in df:
            cols.append(c)
    seen = df[cols].drop_duplicates().copy()
    for _, r in seen.iterrows():
        name = str(r[name_col])
        if any(x["display_name"] == name and x["source_notebook"] == source_notebook for x in config_rows):
            continue
        cid = alias_to_cid.get(name)
        if cid is None and name in reg_lookup:
            cid = reg_lookup[name]["canonical_strategy_id"]
        hist_flag = str(r.get("primary_or_exploratory", "historical")).lower()
        hist = "primary frozen" if hist_flag == "primary" else "exploratory" if hist_flag == "exploratory" else "sensitivity" if hist_flag in {"diagnostic", "sensitivity"} else "secondary robustness"
        constructible = cid is not None or name in set(monthly_strategy_names)
        if cid is None and constructible:
            cid = monthly_strategy_to_cid.get(name, short_hash("monthly|" + name))
        config_rows.append({
            "configuration_id": short_hash(source_notebook + "|" + name + "|" + str(r.get("parameter_values", ""))),
            "display_name": name,
            "source_notebook": source_notebook,
            "source_section": source_section,
            "strategy_family": r.get("strategy_family", ""),
            "strategy_subfamily": r.get("strategy_family", ""),
            "selection_components": "",
            "risk_control_components": "",
            "macro_components": "",
            "leg_allocation_components": "",
            "canonical_components": "",
            "canonical_parameters": r.get("parameter_values", ""),
            "stage_order": "historical source order",
            "parameter_summary": r.get("parameter_values", ""),
            "historical_research_status": hist,
            "historically_primary": hist == "primary frozen",
            "historically_exploratory": hist == "exploratory",
            "historically_excluded": False,
            "historically_sensitivity": hist == "sensitivity",
            "historically_superseded": source_notebook.startswith("06_"),
            "exclusion_reason": "",
            "exclusion_reason_source": "",
            "regression_evidence_status": "",
            "constructible": constructible,
            "construction_status": "monthly source stream available" if constructible else "formula mentioned but no exact persisted stream",
            "nonconstructible_reason": "" if constructible else "not explicitly documented",
            "is_duplicate_alias": cid in set(alias08["canonical_strategy_id"]) if cid else False,
            "duplicate_of_canonical_strategy_id": cid if cid else "",
            "canonical_strategy_id": cid if cid else "",
            "notes": "historical source configuration",
        })

# Normalize monthly return sources before creating historical configs that need hash IDs.
def normalize_returns(df, source_notebook, strategy_col="strategy"):
    if df is None or df.empty or strategy_col not in df:
        return pd.DataFrame()
    d = df.copy()
    d["strategy"] = d[strategy_col].astype(str)
    d["source_notebook"] = source_notebook
    if "variant" in d.columns and strategy_col == "variant":
        d["strategy"] = d["variant"].astype(str)
    d["month_end"] = pd.to_datetime(d["month_end"])
    if "total_gross" in d and "gross_roll_notional" not in d:
        d["gross_roll_notional"] = d["total_gross"]
    for c in ["gross_return", "gross_roll_notional", "long_gross", "short_gross", "net_exposure", "one_way_turnover"]:
        if c not in d:
            d[c] = np.nan
    return d

monthly_sources = [
    normalize_returns(option_returns, "07_option_conditioned_carry_strategy.ipynb"),
    normalize_returns(legacy07_returns, "07_option_conditioned_carry_strategy.ipynb"),
    normalize_returns(old_filter_returns, "06_options_skew_and_vol_filters.ipynb", strategy_col="variant" if "variant" in old_filter_returns.columns else "strategy"),
    normalize_returns(long_side_returns, "06_options_skew_and_vol_filters.ipynb", strategy_col="variant" if "variant" in long_side_returns.columns else "strategy"),
    normalize_returns(readp("08v6_lagged_volatility_benchmark_returns.parquet", required=False), "08_comprehensive_carry_strategy_optimization(6).ipynb"),
]
monthly_returns_all = pd.concat([d for d in monthly_sources if not d.empty], ignore_index=True) if any(not d.empty for d in monthly_sources) else pd.DataFrame()
monthly_strategy_names = sorted(monthly_returns_all["strategy"].dropna().unique().tolist()) if not monthly_returns_all.empty else []

weight_hashes = {}
for wdf, name_col in [(option_weights, "strategy"), (legacy07_weights, "strategy"), (long_side_weights, "variant" if "variant" in long_side_weights.columns else "strategy")]:
    if wdf is None or wdf.empty or name_col not in wdf or "weight" not in wdf:
        continue
    temp = wdf.copy()
    temp["strategy"] = temp[name_col].astype(str)
    temp["month_end"] = pd.to_datetime(temp["month_end"])
    for s, sdf in temp.groupby("strategy"):
        weight_hashes[s] = stable_hash_frame(sdf, ["month_end", "currency", "weight"])

monthly_strategy_to_cid = {}
for s in monthly_strategy_names:
    if s in alias_to_cid:
        monthly_strategy_to_cid[s] = alias_to_cid[s]
    elif s in reg_lookup:
        monthly_strategy_to_cid[s] = reg_lookup[s]["canonical_strategy_id"]
    else:
        sdf = monthly_returns_all[monthly_returns_all["strategy"].eq(s)]
        r_hash = stable_hash_frame(sdf, ["month_end", "gross_return"])
        monthly_strategy_to_cid[s] = short_hash("monthly|" + (weight_hashes.get(s) or r_hash or s))

add_historical_configs(option_returns, "07_option_conditioned_carry_strategy.ipynb", "standalone option strategy returns")
add_historical_configs(legacy07_returns, "07_option_conditioned_carry_strategy.ipynb", "extended historical option sensitivity returns")
add_historical_configs(old_filter_returns, "06_options_skew_and_vol_filters.ipynb", "legacy option/risk filter returns", name_col="variant" if "variant" in old_filter_returns.columns else "strategy")
add_historical_configs(long_side_returns, "06_options_skew_and_vol_filters.ipynb", "long-leg option filter returns", name_col="variant" if "variant" in long_side_returns.columns else "strategy")

configuration_registry = pd.DataFrame(config_rows).drop_duplicates("configuration_id").sort_values(["source_notebook", "display_name"]).reset_index(drop=True)
for col in configuration_registry.select_dtypes(include=["object"]).columns:
    configuration_registry[col] = configuration_registry[col].fillna("").astype(str)
configuration_registry.to_parquet(OUTPUTS["configuration_registry"], index=False)
display_df("Configuration registry sample", configuration_registry, n=20)

**Configuration registry sample**

,configuration_id,display_name,source_notebook,source_section,strategy_family,strategy_subfamily,selection_components,risk_control_components,macro_components,leg_allocation_components,canonical_components,canonical_parameters,stage_order,parameter_summary,historical_research_status,historically_primary,historically_exploratory,historically_excluded,historically_sensitivity,historically_superseded,exclusion_reason,exclusion_reason_source,regression_evidence_status,constructible,construction_status,nonconstructible_reason,is_duplicate_alias,duplicate_of_canonical_strategy_id,canonical_strategy_id,notes
0,93548969aab3fed28167,EM-long bad-skew scaling lambda=0.5,06_options_skew_and_vol_filters.ipynb,long-leg option filter returns,,,,,,,,,historical source order,,secondary robustness,False,False,False,False,True,,,,True,monthly source stream available,,False,b4eca8d318cf9321de72,b4eca8d318cf9321de72,historical source configuration
1,7c164b58390f7df6677b,EM-long-only bad-skew exclusion q80,06_options_skew_and_vol_filters.ipynb,long-leg option filter returns,,,,,,,,,historical source order,,secondary robustness,False,False,False,False,True,,,,True,monthly source stream available,,False,611069a575cb96857919,611069a575cb96857919,historical source configuration
2,63f882804ef9068294af,EM-long-only bad-skew exclusion q90,06_options_skew_and_vol_filters.ipynb,long-leg option filter returns,,,,,,,,,historical source order,,secondary robustness,False,False,False,False,True,,,,True,monthly source stream available,,False,0d1f3fe2ebf033cb87a3,0d1f3fe2ebf033cb87a3,historical source configuration
3,468342c16750213c4891,EM-long-score skew lambda=0.5,06_options_skew_and_vol_filters.ipynb,long-leg option filter returns,,,,,,,,,historical source order,,secondary robustness,False,False,False,False,True,,,,True,monthly source stream available,,False,d36821521c1be4f2f9b2,d36821521c1be4f2f9b2,historical source configuration
4,048aacec97248d49ef5d,bad-skew exclusion,06_options_skew_and_vol_filters.ipynb,legacy option/risk filter returns,,,,,,,,,historical source order,,secondary robustness,False,False,False,False,True,,,,True,monthly source stream available,,False,232dcd540fba6166e68b,232dcd540fba6166e68b,historical source configuration
5,0eb3268af24e510a9fc1,baseline,06_options_skew_and_vol_filters.ipynb,legacy option/risk filter returns,,,,,,,,,historical source order,,secondary robustness,False,False,False,False,True,,,,True,monthly source stream available,,False,274b6fd8b98e2252d035,274b6fd8b98e2252d035,historical source configuration
6,31b52510f025e2c9baf0,baseline + global risk-off 50pct,06_options_skew_and_vol_filters.ipynb,legacy option/risk filter returns,,,,,,,,,historical source order,,secondary robustness,False,False,False,False,True,,,,True,monthly source stream available,,False,0760f0a3f4a22131cc94,0760f0a3f4a22131cc94,historical source configuration
7,54df73d216918a0c910e,baseline + global risk-off zero,06_options_skew_and_vol_filters.ipynb,legacy option/risk filter returns,,,,,,,,,historical source order,,secondary robustness,False,False,False,False,True,,,,True,monthly source stream available,,False,56244e580c976a368858,56244e580c976a368858,historical source configuration
8,f734268dfb5d25c76ee1,de-risk option vol target 10pct,06_options_skew_and_vol_filters.ipynb,legacy option/risk filter returns,,,,,,,,,historical source order,,secondary robustness,False,False,False,False,True,,,,True,monthly source stream available,,False,dfb4913b72788797d325,dfb4913b72788797d325,historical source configuration
9,bd1862a4a8166578602a,de-risk option vol target 10pct + global risk-off 50pct,06_options_skew_and_vol_filters.ipynb,legacy option/risk filter returns,,,,,,,,,historical source order,,secondary robustness,False,False,False,False,True,,,,True,monthly source stream available,,False,8269ce3be205c205bded,8269ce3be205c205bded,historical source configuration


## 4. Canonical Identity and Deduplication

Canonical IDs are unique strategy streams. For Notebook 08 strategies, the saved canonical IDs and weight hashes are authoritative. Historical Notebook 07/06 streams that are not already in Notebook 08 are added as additional canonical rows and linked back to their configurations.

In [5]:
monthly_perf_rows = []
if not monthly_returns_all.empty:
    for (source_nb, strategy), sdf in monthly_returns_all.groupby(["source_notebook", "strategy"]):
        stats = perf_stats(sdf, "gross_return")
        stats.update({
            "source_notebook": source_nb,
            "strategy": strategy,
            "return_stream_hash": stable_hash_frame(sdf, ["month_end", "gross_return"]),
            "weight_stream_hash": weight_hashes.get(strategy, ""),
        })
        for bp in [1, 2, 5, 10, 25, 50]:
            col = f"net_return_{bp}bp"
            if col in sdf:
                stats[f"net_sharpe_{bp}bp"] = perf_stats(sdf, col)["gross_sharpe"]
                stats[f"net_annualized_return_{bp}bp"] = perf_stats(sdf, col)["annualized_arithmetic_return"]
            elif f"net_return_{bp}" in sdf:
                stats[f"net_sharpe_{bp}bp"] = perf_stats(sdf, f"net_return_{bp}")["gross_sharpe"]
                stats[f"net_annualized_return_{bp}bp"] = perf_stats(sdf, f"net_return_{bp}")["annualized_arithmetic_return"]
        monthly_perf_rows.append(stats)
monthly_performance = pd.DataFrame(monthly_perf_rows)

canonical_rows = []
for _, r in reg08.iterrows():
    strategy = str(r["strategy"])
    cls = class_lookup.get(strategy, {})
    dec = decision08[decision08["Strategy"].astype(str).eq(strategy)].head(1)
    ev = eval08[eval08["Strategy"].astype(str).eq(strategy)].head(1)
    cost = cost08[cost08["Strategy"].astype(str).eq(strategy)].head(1)
    mperf = monthly_performance[monthly_performance["strategy"].astype(str).eq(strategy)].head(1) if not monthly_performance.empty else pd.DataFrame()
    ret_hash = mperf["return_stream_hash"].iloc[0] if not mperf.empty else short_hash("08-performance|" + strategy + "|" + str(dec.to_dict("records")))
    canonical_rows.append({
        "canonical_strategy_id": r["canonical_strategy_id"],
        "canonical_display_name": r.get("canonical_display_name", strategy),
        "canonical_components": r.get("option_architecture", ""),
        "canonical_parameters": r.get("parameter_signature", ""),
        "stage_order": r.get("stage_sequence", ""),
        "strategy_family": r.get("strategy_family", ""),
        "strategy_aliases": alias08.loc[alias08["canonical_strategy_id"].eq(r["canonical_strategy_id"]), "alias"].astype(str).tolist() if not alias08.empty else [strategy],
        "configuration_count": int(configuration_registry["canonical_strategy_id"].astype(str).eq(str(r["canonical_strategy_id"])).sum()),
        "weight_stream_hash": r.get("weight_stream_hash", ""),
        "return_stream_hash": ret_hash,
        "source_notebook": "08_comprehensive_carry_strategy_optimization(6).ipynb",
        "historical_research_status": status_from_role(cls),
        "constructible": True,
        "performance_available": not dec.empty,
        "monthly_return_stream_available": not mperf.empty,
    })

existing_cids = {str(x) for x in reg08["canonical_strategy_id"]}
for s, cid in monthly_strategy_to_cid.items():
    if str(cid) in existing_cids:
        continue
    mperf = monthly_performance[monthly_performance["strategy"].astype(str).eq(s)].head(1)
    canonical_rows.append({
        "canonical_strategy_id": cid,
        "canonical_display_name": s,
        "canonical_components": "",
        "canonical_parameters": "",
        "stage_order": "historical source order",
        "strategy_family": mperf["source_notebook"].iloc[0] if not mperf.empty else "historical monthly stream",
        "strategy_aliases": [s],
        "configuration_count": int(configuration_registry["canonical_strategy_id"].astype(str).eq(str(cid)).sum()),
        "weight_stream_hash": weight_hashes.get(s, ""),
        "return_stream_hash": mperf["return_stream_hash"].iloc[0] if not mperf.empty else "",
        "source_notebook": mperf["source_notebook"].iloc[0] if not mperf.empty else "",
        "historical_research_status": "historical monthly stream",
        "constructible": True,
        "performance_available": not mperf.empty,
        "monthly_return_stream_available": not mperf.empty,
    })

canonical_strategy_registry = pd.DataFrame(canonical_rows).sort_values(["source_notebook", "canonical_display_name"]).reset_index(drop=True)
canonical_collision_audit = canonical_strategy_registry[
    canonical_strategy_registry["canonical_strategy_id"].duplicated(keep=False)
].copy()
if not canonical_collision_audit.empty:
    print(
        f"WARNING: {canonical_collision_audit['canonical_strategy_id'].nunique()} canonical IDs "
        "appeared more than once across source notebooks. Keeping one canonical row per ID; "
        "all historical names remain visible in the configuration registry."
    )
canonical_strategy_registry = canonical_strategy_registry.drop_duplicates(
    "canonical_strategy_id", keep="first"
).reset_index(drop=True)
assert canonical_strategy_registry["canonical_strategy_id"].is_unique

configuration_registry["is_duplicate_alias"] = configuration_registry.duplicated("canonical_strategy_id", keep="first") & configuration_registry["canonical_strategy_id"].astype(str).ne("")
dup_sizes = configuration_registry.groupby("canonical_strategy_id")["configuration_id"].transform("count")
configuration_alias_audit = configuration_registry[[
    "configuration_id", "display_name", "canonical_strategy_id", "is_duplicate_alias"
]].copy()
configuration_alias_audit["weight_hash"] = configuration_alias_audit["canonical_strategy_id"].map(canonical_strategy_registry.set_index("canonical_strategy_id")["weight_stream_hash"])
configuration_alias_audit["return_hash"] = configuration_alias_audit["canonical_strategy_id"].map(canonical_strategy_registry.set_index("canonical_strategy_id")["return_stream_hash"])
configuration_alias_audit["duplicate_group_size"] = dup_sizes
configuration_alias_audit["is_canonical_representative"] = ~configuration_registry.duplicated("canonical_strategy_id", keep="first")

canonical_strategy_registry.to_parquet(OUTPUTS["canonical_strategy_registry"], index=False)
configuration_alias_audit.to_parquet(OUTPUTS["configuration_alias_audit"], index=False)
display_df("Canonical strategy registry sample", canonical_strategy_registry, n=20)
display_df("Configuration alias audit sample", configuration_alias_audit.sort_values("duplicate_group_size", ascending=False), n=20)

**Canonical strategy registry sample**

,canonical_strategy_id,canonical_display_name,canonical_components,canonical_parameters,stage_order,strategy_family,strategy_aliases,configuration_count,weight_stream_hash,return_stream_hash,source_notebook,historical_research_status,constructible,performance_available,monthly_return_stream_available
0,b4eca8d318cf9321de72,EM-long bad-skew scaling lambda=0.5,,,historical source order,06_options_skew_and_vol_filters.ipynb,['EM-long bad-skew scaling lambda=0.5'],1,fd2b9e794f46416e867a0513087ed1e2de66360ad51262a0d1682e31819288e3,60d9020968f84af3fc6cb1a89e1d3c3fe8b70eb4aad66ee596bcf6d016396431,06_options_skew_and_vol_filters.ipynb,historical monthly stream,True,True,True
1,611069a575cb96857919,EM-long-only bad-skew exclusion q80,,,historical source order,06_options_skew_and_vol_filters.ipynb,['EM-long-only bad-skew exclusion q80'],1,95e306d4f7107d4cd815af00bf2b2f35fc14f4a2cd999b18bba418cb34f8d475,3f00ae352e9ebdc13551c28b6a6ef006d72c36719b70f0f8c1a88bc77bdcd785,06_options_skew_and_vol_filters.ipynb,historical monthly stream,True,True,True
2,0d1f3fe2ebf033cb87a3,EM-long-only bad-skew exclusion q90,,,historical source order,06_options_skew_and_vol_filters.ipynb,['EM-long-only bad-skew exclusion q90'],1,92d42414725d1c013512ed89289366b48470ced9048c13f81236f77770e11669,5556a7155aa0e1364c2210d9fddda73b4b614e690279002abdfd0e91e3037a8e,06_options_skew_and_vol_filters.ipynb,historical monthly stream,True,True,True
3,d36821521c1be4f2f9b2,EM-long-score skew lambda=0.5,,,historical source order,06_options_skew_and_vol_filters.ipynb,['EM-long-score skew lambda=0.5'],1,05870bae5de2169c8ee72afb3399ebc42ca9942d51b8babc30f35466e63086fe,2340b0e1fbf774d0aca828018c019c82aa4c643f1f138fb3c39fc211890ff02a,06_options_skew_and_vol_filters.ipynb,historical monthly stream,True,True,True
4,232dcd540fba6166e68b,bad-skew exclusion,,,historical source order,06_options_skew_and_vol_filters.ipynb,['bad-skew exclusion'],1,,49be90aa2b0eb107698e1e9c6eb7b260829bae4b4f97450c37d8ef2f6af1d3f2,06_options_skew_and_vol_filters.ipynb,historical monthly stream,True,True,True
5,274b6fd8b98e2252d035,baseline,,,historical source order,06_options_skew_and_vol_filters.ipynb,['baseline'],1,,de5b4f1ed7f87332e539844197718116d1a6870dc89e9041cf0935dfe6c08390,06_options_skew_and_vol_filters.ipynb,historical monthly stream,True,True,True
6,0760f0a3f4a22131cc94,baseline + global risk-off 50pct,,,historical source order,06_options_skew_and_vol_filters.ipynb,['baseline + global risk-off 50pct'],1,,7b9e3e57869b43d9dfeac11a921f4aabbd2f91e124c5e79fa221a222602e2d3e,06_options_skew_and_vol_filters.ipynb,historical monthly stream,True,True,True
7,56244e580c976a368858,baseline + global risk-off zero,,,historical source order,06_options_skew_and_vol_filters.ipynb,['baseline + global risk-off zero'],1,,6826c082bb23260baa59010d2f4a422b6b645612d68799cc6b0e8c09d4f20e4b,06_options_skew_and_vol_filters.ipynb,historical monthly stream,True,True,True
8,dfb4913b72788797d325,de-risk option vol target 10pct,,,historical source order,06_options_skew_and_vol_filters.ipynb,['de-risk option vol target 10pct'],1,,9f27ed42889c251c2b57bdd2ad26ffcc733d2289ec077e74d0ecb9398248c5b1,06_options_skew_and_vol_filters.ipynb,historical monthly stream,True,True,True
9,8269ce3be205c205bded,de-risk option vol target 10pct + global risk-off 50pct,,,historical source order,06_options_skew_and_vol_filters.ipynb,['de-risk option vol target 10pct + global risk-off 50pct'],1,,e6c481da95d2813bf479280f289cc32b0031f08e7e73023a01f7ccfbcea54078,06_options_skew_and_vol_filters.ipynb,historical monthly stream,True,True,True


**Configuration alias audit sample**

,configuration_id,display_name,canonical_strategy_id,is_duplicate_alias,weight_hash,return_hash,duplicate_group_size,is_canonical_representative
161,9735f8b42de2fb84507d,Baseline carry + SPX_tail_return_1m,bb4abd1af9d7add2b610,True,406eae49214c576fd8295213abd9efd7312cacf7b41ff75bea846575b1763224,1d064ee4a40f25297579,4,False
150,ed35cc57195cafb0bd97,Baseline carry + MXEF_tail_return_1m,ce39d844cc839d9f7316,True,273f91855db18977cafb499b1f2c8a61b25fc98fafad330250fc27d5f15bb379,1b509b70716b74883807,4,False
400,4505b5f6a84a09980d0b,MXEF_tail_return_1m gross scaling 50%,ce39d844cc839d9f7316,True,273f91855db18977cafb499b1f2c8a61b25fc98fafad330250fc27d5f15bb379,1b509b70716b74883807,4,False
398,3543ec3ca184e4928f95,MXEF_return_1m gross scaling 50%,ce39d844cc839d9f7316,True,273f91855db18977cafb499b1f2c8a61b25fc98fafad330250fc27d5f15bb379,1b509b70716b74883807,4,False
415,451ba9ebab304302e349,SPX_return_1m gross scaling 50%,bb4abd1af9d7add2b610,True,406eae49214c576fd8295213abd9efd7312cacf7b41ff75bea846575b1763224,1d064ee4a40f25297579,4,False
417,1f3909884b8775537018,SPX_tail_return_1m gross scaling 50%,bb4abd1af9d7add2b610,True,406eae49214c576fd8295213abd9efd7312cacf7b41ff75bea846575b1763224,1d064ee4a40f25297579,4,False
149,0787f5b28957b29f3bae,Baseline carry + MXEF_return_1m,ce39d844cc839d9f7316,False,273f91855db18977cafb499b1f2c8a61b25fc98fafad330250fc27d5f15bb379,1b509b70716b74883807,4,True
160,1cbd0197a4bf5c257fd6,Baseline carry + SPX_return_1m,bb4abd1af9d7add2b610,False,406eae49214c576fd8295213abd9efd7312cacf7b41ff75bea846575b1763224,1d064ee4a40f25297579,4,True
50,72a1110265505d6705e2,Baseline carry,d2c40321592906dca3c6,False,a42de5c8f46292f9431e677627743af99c8ffde6f781284c902bf40f643dcc77,7fa7cd273e7ebadec564c9c012b9e7176dd716cd433190cf07793255add2f215,3,True
152,e00c75e9fb2d656e0752,Baseline carry + OIL_return_1m,a12cb0f36d9810feba65,False,293b3332ac7591b9656e9606156cd5daa2b7efa707ac82a334d076eb45f728e5,953f49aa02f856935347,3,True


## 5. Unified Performance Engine and Master Leaderboards

Notebook 09 uses one reporting schema. Where full monthly return streams are persisted, the metrics are recomputed directly. For Notebook 08 integrated strategies without monthly streams, the notebook carries forward Notebook 08's authoritative common-engine metrics and records the source limitation explicitly.

In [6]:
full = decision08.rename(columns={
    "Strategy": "strategy",
    "Reference": "reference_strategy",
    "Annualized return": "full_return",
    "Gross Sharpe": "full_sharpe",
    "Maximum drawdown": "full_mdd",
    "Expected shortfall": "full_es",
    "Average gross": "average_total_gross",
    "Average broad-regime annualized monthly delta": "average_broad_regime_delta",
    "Average fixed-crisis annualized monthly delta": "average_fixed_crisis_delta",
    "Net Sharpe at 5bp": "net_sharpe_5bp",
}).copy()
if "Annualized volatility" in full:
    full["full_vol"] = full["Annualized volatility"]
else:
    full["full_vol"] = np.where(full["full_sharpe"].replace(0, np.nan).notna(), full["full_return"] / full["full_sharpe"], np.nan)

evalp = eval08.rename(columns={
    "Strategy": "strategy",
    "Annualized return": "evaluation_return",
    "Volatility": "evaluation_vol",
    "Gross Sharpe": "evaluation_sharpe",
    "Maximum drawdown": "evaluation_mdd",
    "Expected shortfall": "evaluation_es",
    "Delta Sharpe vs reference": "evaluation_delta_sharpe_vs_reference",
    "Average one-way turnover": "evaluation_turnover",
}).copy()
cost = cost08.rename(columns={"Strategy": "strategy"}).copy()
cls = class08.rename(columns={"strategy": "strategy"}).copy()

leader = canonical_strategy_registry.rename(columns={"canonical_display_name": "strategy"}).copy()
leader = leader.merge(full, on="strategy", how="left", suffixes=("", "_full"))
leader = leader.merge(evalp[["strategy", "evaluation_return", "evaluation_vol", "evaluation_sharpe", "evaluation_mdd", "evaluation_es", "evaluation_delta_sharpe_vs_reference", "evaluation_turnover"]], on="strategy", how="left")
leader = leader.merge(cost, on="strategy", how="left", suffixes=("", "_cost"))
leader = leader.merge(cls[["strategy", "label", "classification_tier", "evaluation_role", "point_in_time_status", "future_horizon_leakage", "multiple_testing_strength", "BH_adjusted_p_value"]], on="strategy", how="left")

# Add any historical monthly-only canonical strategies absent from Notebook 08.
extra_perf = monthly_performance[~monthly_performance["strategy"].isin(set(leader["strategy"]))].copy() if not monthly_performance.empty else pd.DataFrame()
if not extra_perf.empty:
    extra = extra_perf.rename(columns={
        "annualized_arithmetic_return": "full_return",
        "annualized_volatility": "full_vol",
        "gross_sharpe": "full_sharpe",
        "maximum_drawdown": "full_mdd",
        "expected_shortfall_5pct": "full_es",
        "average_total_gross": "average_total_gross",
    })
    extra["reference_strategy"] = "Baseline carry"
    extra["canonical_strategy_id"] = extra["strategy"].map(monthly_strategy_to_cid)
    existing_leader_cids = set(leader["canonical_strategy_id"].astype(str))
    extra = extra[~extra["canonical_strategy_id"].astype(str).isin(existing_leader_cids)].copy()
    extra["canonical_components"] = ""
    extra["canonical_parameters"] = ""
    extra["stage_order"] = "historical source order"
    extra["strategy_aliases"] = extra["strategy"].map(lambda x: [x])
    extra["configuration_count"] = extra["canonical_strategy_id"].map(configuration_registry.groupby("canonical_strategy_id").size())
    extra["weight_stream_hash"] = extra["strategy"].map(weight_hashes).fillna("")
    extra["source_notebook"] = extra["source_notebook"]
    extra["historical_research_status"] = "historical monthly stream"
    extra["constructible"] = True
    extra["performance_available"] = True
    extra["monthly_return_stream_available"] = True
    extra["label"] = "historical strategy"
    extra["classification_tier"] = "Historical / not ranked by Notebook 08"
    extra["evaluation_role"] = "historical source strategy"
    if not extra.empty:
        leader = pd.concat([leader, extra.reindex(columns=leader.columns)], ignore_index=True)

leader["full_es"] = leader["full_es"].fillna(leader.get("Expected shortfall", np.nan))
def column_or_nan(df, col):
    return df[col] if col in df.columns else pd.Series(np.nan, index=df.index)

leader["average_one_way_turnover"] = column_or_nan(leader, "Average one-way turnover").fillna(
    column_or_nan(leader, "average_one_way_turnover")
)
leader["average_total_gross"] = column_or_nan(leader, "average_total_gross").fillna(
    column_or_nan(leader, "Average gross")
)
leader["evaluation_delta_sharpe_vs_baseline"] = leader["evaluation_sharpe"] - leader.loc[leader["strategy"].eq("Baseline carry"), "evaluation_sharpe"].iloc[0]
parent_sharpe = leader.set_index("strategy")["evaluation_sharpe"].to_dict()
leader["evaluation_delta_sharpe_vs_parent"] = leader["evaluation_sharpe"] - leader["reference_strategy"].map(parent_sharpe)
for bp in [1, 2, 5, 10, 25, 50]:
    src = f"Net Sharpe {bp}bp"
    if src in leader:
        leader[f"net_sharpe_{bp}bp"] = leader[src]
leader["rank_full_sharpe"] = leader["full_sharpe"].rank(ascending=False, method="min")
leader["rank_evaluation_sharpe"] = leader["evaluation_sharpe"].rank(ascending=False, method="min")
leader["rank_annualized_return"] = leader["full_return"].rank(ascending=False, method="min")
leader["rank_lowest_volatility"] = leader["full_vol"].rank(ascending=True, method="min")
leader["rank_best_mdd"] = leader["full_mdd"].rank(ascending=False, method="min")
leader["rank_best_es"] = leader["full_es"].rank(ascending=False, method="min")
leader["rank_lowest_turnover"] = leader["average_one_way_turnover"].rank(ascending=True, method="min")
leader["rank_5bp_net_sharpe"] = leader["net_sharpe_5bp"].rank(ascending=False, method="min") if "net_sharpe_5bp" in leader else np.nan
leader["rank_delta_sharpe_vs_parent"] = leader["evaluation_delta_sharpe_vs_parent"].rank(ascending=False, method="min")

canonical_strategy_leaderboard = leader.sort_values("rank_full_sharpe", na_position="last").reset_index(drop=True)
canonical_strategy_leaderboard.to_parquet(OUTPUTS["canonical_strategy_leaderboard"], index=False)

config_results = configuration_registry.merge(
    canonical_strategy_leaderboard.drop(columns=["strategy"], errors="ignore"),
    on="canonical_strategy_id",
    how="left",
    suffixes=("", "_canonical"),
)
config_results["performance_metrics_available"] = config_results["constructible"] & config_results["full_sharpe"].notna()
config_results.to_parquet(OUTPUTS["all_configuration_results"], index=False)

display_df("Canonical leaderboard: full-sample Sharpe", canonical_strategy_leaderboard[["strategy", "strategy_family", "historical_research_status", "classification_tier", "full_return", "full_sharpe", "full_mdd", "full_es", "average_total_gross", "net_sharpe_5bp"]], n=15, fmt={"full_return":"{:.2%}","full_sharpe":"{:.2f}","full_mdd":"{:.2%}","full_es":"{:.2%}","average_total_gross":"{:.2f}","net_sharpe_5bp":"{:.2f}"})

**Canonical leaderboard: full-sample Sharpe**

,strategy,strategy_family,historical_research_status,classification_tier,full_return,full_sharpe,full_mdd,full_es,average_total_gross,net_sharpe_5bp
0,A+C x Continuous macro-risk rule,ultimate exploratory matrix,exploratory,Tier 3 exploratory watchlist,2.99%,0.75,-11.57%,-2.51%,1.02,0.56
1,Core A+B + MXEF_momentum_3m,macro extension,exploratory,Tier 2 promising candidate,5.15%,0.74,-17.51%,-4.22%,1.80,0.56
2,Core A+C + MXEF_momentum_3m,macro extension,exploratory,Tier 2 promising candidate,3.00%,0.74,-11.57%,-2.54%,1.02,0.56
3,Core A+B + SPX_momentum_3m,macro extension,exploratory,Tier 2 promising candidate,5.25%,0.74,-17.51%,-4.45%,1.80,0.56
4,Core A+C + COMMODITY_momentum_3m,macro extension,exploratory,Tier 3 exploratory watchlist,2.91%,0.74,-11.57%,-2.56%,1.01,0.55
5,Core A+C + MOVE_change_1m,macro extension,exploratory,Tier 3 exploratory watchlist,2.99%,0.74,-11.57%,-2.53%,1.03,0.55
6,Core A+C + MXEF_return_1m,macro extension,exploratory,Tier 3 exploratory watchlist,3.03%,0.74,-11.57%,-2.63%,1.03,0.55
7,Core A+C + SPX_momentum_3m,macro extension,exploratory,Tier 2 promising candidate,2.96%,0.74,-11.57%,-2.55%,1.02,0.55
8,Core A+B + OIL_return_1m,macro extension,exploratory,Tier 3 exploratory watchlist,5.19%,0.74,-17.51%,-4.30%,1.80,0.55
9,Core A+B + MXEF_return_1m,macro extension,exploratory,Tier 3 exploratory watchlist,5.31%,0.73,-17.51%,-4.62%,1.83,0.55


## 6. Full / Discovery / Validation / Evaluation Results and Costs

The frozen chronological sample windows are inherited from Notebook 08. Monthly return streams are used where available; otherwise Notebook 08's persisted sample metrics remain the authoritative source.

In [7]:
split_dates = {
    "Full": tuple(summary08.get("full_sample_dates", [None, None])),
    "Discovery": tuple(summary08.get("discovery_dates", [None, None])),
    "Validation": tuple(summary08.get("validation_dates", [None, None])),
    "Frozen Evaluation": tuple(summary08.get("evaluation_dates", [None, None])),
}

sample_perf_rows = []
if not monthly_returns_all.empty:
    for (source_nb, strategy), sdf in monthly_returns_all.groupby(["source_notebook", "strategy"]):
        for sample_name, (start, end) in split_dates.items():
            if not start or not end:
                continue
            sub = sdf[(sdf["month_end"] >= pd.Timestamp(start)) & (sdf["month_end"] <= pd.Timestamp(end))]
            stats = perf_stats(sub, "gross_return")
            stats.update({"strategy": strategy, "source_notebook": source_nb, "sample_window": sample_name, "sample_start": start, "sample_end": end})
            sample_perf_rows.append(stats)
sample_performance = pd.DataFrame(sample_perf_rows)
sample_performance.to_parquet(OUTPUTS["sample_performance"], index=False)

cost_sensitivity = canonical_strategy_leaderboard[["canonical_strategy_id", "strategy"]].merge(cost08.rename(columns={"Strategy": "strategy"}), on="strategy", how="left")
for bp in [1, 2, 5, 10, 25, 50]:
    net = f"Net Sharpe {bp}bp"
    if net in cost_sensitivity:
        cost_sensitivity[f"rank_net_sharpe_{bp}bp"] = cost_sensitivity[net].rank(ascending=False, method="min")
cost_sensitivity.to_parquet(OUTPUTS["cost_sensitivity"], index=False)

display_df("Sample performance from available monthly streams", sample_performance.sort_values(["sample_window", "gross_sharpe"], ascending=[True, False]), n=20, fmt={"annualized_arithmetic_return":"{:.2%}","annualized_volatility":"{:.2%}","gross_sharpe":"{:.2f}","maximum_drawdown":"{:.2%}"})
display_df("Cost sensitivity sample", cost_sensitivity.sort_values("Net Sharpe 5bp", ascending=False, na_position="last")[["strategy","Gross Sharpe","Net Sharpe 1bp","Net Sharpe 5bp","Net Sharpe 10bp","Net Sharpe 25bp","Net Sharpe 50bp"]], n=20, fmt={"Gross Sharpe":"{:.2f}","Net Sharpe 1bp":"{:.2f}","Net Sharpe 5bp":"{:.2f}","Net Sharpe 10bp":"{:.2f}","Net Sharpe 25bp":"{:.2f}","Net Sharpe 50bp":"{:.2f}"})

**Sample performance from available monthly streams**

,months,annualized_arithmetic_return,annualized_volatility,gross_sharpe,sortino_ratio,maximum_drawdown,expected_shortfall_5pct,hit_rate,skewness,worst_month,best_month,final_nav,average_long_gross,average_short_gross,average_total_gross,average_net_exposure,average_one_way_turnover,long_leg_annualized_contribution,short_leg_annualized_contribution,strategy,source_notebook,sample_window,sample_start,sample_end
29,140,3.19%,7.39%,0.43,0.495430,-12.26%,-0.049801,0.407143,-0.442463,2018-07-31 00:00:00,2008-06-30 00:00:00,1.404778,,,1.400000,,,0.000000,0.000000,baseline + global risk-off zero,06_options_skew_and_vol_filters.ipynb,Discovery,2007-01-31,2018-08-31
1,140,3.71%,8.62%,0.43,0.589330,-16.42%,-0.057904,0.585714,-0.533422,2008-09-30 00:00:00,2017-01-31 00:00:00,1.475561,1.000000,1.000000,2.000000,,,0.000000,0.000000,EM-long bad-skew scaling lambda=0.5,06_options_skew_and_vol_filters.ipynb,Discovery,2007-01-31,2018-08-31
157,280,2.32%,5.64%,0.41,0.621144,-15.36%,-0.035351,0.550000,-0.190211,2008-09-30 00:00:00,2008-10-31 00:00:00,1.653829,1.000000,1.000000,2.000000,,,0.000000,0.000000,"long-score vol-skew lambda=1.0,gamma=1.0",06_options_skew_and_vol_filters.ipynb,Discovery,2007-01-31,2018-08-31
25,140,3.26%,8.02%,0.41,0.549436,-12.57%,-0.052783,0.592857,-0.490629,2018-07-31 00:00:00,2008-06-30 00:00:00,1.409024,,,1.700000,,,0.000000,0.000000,baseline + global risk-off 50pct,06_options_skew_and_vol_filters.ipynb,Discovery,2007-01-31,2018-08-31
333,420,3.11%,7.65%,0.41,0.572044,-35.47%,-0.049824,0.550000,-0.611961,2008-09-30 00:00:00,2009-02-28 00:00:00,2.679341,1.000000,1.000000,2.000000,-0.000000,0.302019,0.000000,0.000000,Currency-level neutral-missing Skew risk scaling fixed-gross gamma=0.5,07_option_conditioned_carry_strategy.ipynb,Discovery,2007-01-31,2018-08-31
337,420,2.33%,5.97%,0.39,0.551333,-28.29%,-0.039201,0.550000,-0.602249,2008-09-30 00:00:00,2009-02-28 00:00:00,2.124614,0.788962,0.788962,1.577923,-0.000000,0.249918,0.000000,0.000000,Currency-level neutral-missing Skew risk scaling variable-gross gamma=0.5,07_option_conditioned_carry_strategy.ipynb,Discovery,2007-01-31,2018-08-31
45,140,2.28%,5.84%,0.39,0.507002,-9.88%,-0.038710,0.592857,-0.590699,2018-07-31 00:00:00,2007-04-30 00:00:00,1.279095,,,1.253857,,,0.000000,0.000000,de-risk option vol target 8pct + global risk-off 50pct,06_options_skew_and_vol_filters.ipynb,Discovery,2007-01-31,2018-08-31
265,420,2.17%,5.58%,0.39,0.569084,-24.98%,-0.035092,0.554762,-0.381894,2008-09-30 00:00:00,2009-02-28 00:00:00,2.026050,0.699312,0.699312,1.398624,-0.000000,0.210711,0.000000,0.000000,Coverage-aware Butterfly risk scaling gamma=1,07_option_conditioned_carry_strategy.ipynb,Discovery,2007-01-31,2018-08-31
261,420,2.34%,6.03%,0.39,0.561258,-27.02%,-0.038185,0.554762,-0.439767,2008-09-30 00:00:00,2009-02-28 00:00:00,2.125851,0.754455,0.754455,1.508910,-0.000000,0.217803,0.000000,0.000000,Coverage-aware Butterfly risk scaling gamma=0.75,07_option_conditioned_carry_strategy.ipynb,Discovery,2007-01-31,2018-08-31
257,420,2.53%,6.58%,0.39,0.552436,-29.44%,-0.041946,0.554762,-0.498606,2008-09-30 00:00:00,2009-02-28 00:00:00,2.249240,0.820205,0.820205,1.640410,-0.000000,0.224939,0.000000,0.000000,Coverage-aware Butterfly risk scaling gamma=0.5,07_option_conditioned_carry_strategy.ipynb,Discovery,2007-01-31,2018-08-31


**Cost sensitivity sample**

,strategy,Gross Sharpe,Net Sharpe 1bp,Net Sharpe 5bp,Net Sharpe 10bp,Net Sharpe 25bp,Net Sharpe 50bp
0,A+C x Continuous macro-risk rule,0.75,0.71,0.56,0.37,-0.21,-1.17
2,Core A+C + MXEF_momentum_3m,0.74,0.71,0.56,0.37,-0.18,-1.12
3,Core A+B + SPX_momentum_3m,0.74,0.71,0.56,0.37,-0.18,-1.10
1,Core A+B + MXEF_momentum_3m,0.74,0.71,0.56,0.37,-0.19,-1.14
4,Core A+C + COMMODITY_momentum_3m,0.74,0.70,0.55,0.36,-0.20,-1.14
19,G10-aware ATM conditioning g10=0.5 em=0 + MXEF_return_1m,0.71,0.68,0.55,0.39,-0.08,-0.87
7,Core A+C + SPX_momentum_3m,0.74,0.70,0.55,0.37,-0.19,-1.13
6,Core A+C + MXEF_return_1m,0.74,0.70,0.55,0.36,-0.20,-1.15
5,Core A+C + MOVE_change_1m,0.74,0.70,0.55,0.36,-0.21,-1.16
8,Core A+B + OIL_return_1m,0.74,0.70,0.55,0.36,-0.21,-1.15


## 7. A/B/C/D Factorial Universe

The factorial uses documented Notebook 08 definitions:

- A: G10-aware ATM-conditioned ranking / selection;
- B: butterfly-conditioned ranking / selection;
- C: joint option-risk variable-gross scaling;
- D: frozen primary macro overlay.

All 16 states are resolved to exact stored Notebook 08 strategy names. The table is descriptive and not a new search.

In [8]:
ABCD_STRATEGIES = {
    "0000": "Baseline carry",
    "1000": "G10-aware ATM conditioning g10=0.5 em=0",
    "0100": "Butterfly-conditioned carry lambda=-0.5",
    "0010": "Joint option-risk scaling variable-gross gamma=0.5",
    "0001": "OIL_return_1m gross scaling 50%",
    "1100": "Core A+B",
    "1010": "Core A+C",
    "1001": "G10-aware ATM conditioning g10=0.5 em=0 + OIL_return_1m",
    "0110": "Core B+C",
    "0101": "Butterfly-conditioned carry lambda=-0.5 + OIL_return_1m",
    "0011": "Joint option-risk scaling variable-gross gamma=0.5 + OIL_return_1m",
    "1110": "Core A+B+C",
    "1101": "Core A+B + OIL_return_1m",
    "1011": "Core A+C + OIL_return_1m",
    "0111": "Core B+C + OIL_return_1m",
    "1111": "Core A+B+C + OIL_return_1m",
}
reg_names = set(reg08["strategy"].astype(str))
abcd_rows = []
for bits, strat in ABCD_STRATEGIES.items():
    row = {
        "abcd_state": bits,
        "A": int(bits[0]), "B": int(bits[1]), "C": int(bits[2]), "D": int(bits[3]),
        "strategy": strat,
        "constructible": strat in reg_names,
        "construction_status": "resolved to Notebook 08 canonical strategy" if strat in reg_names else "missing exact Notebook 08 strategy name",
    }
    if strat in reg_lookup:
        row["canonical_strategy_id"] = reg_lookup[strat]["canonical_strategy_id"]
    abcd_rows.append(row)
abcd_factorial_registry = pd.DataFrame(abcd_rows)
if not abcd_factorial_registry["constructible"].all():
    raise AssertionError("Missing ABCD states: " + abcd_factorial_registry.loc[~abcd_factorial_registry["constructible"], ["abcd_state", "strategy"]].to_string(index=False))
abcd_perf = abcd_factorial_registry.merge(canonical_strategy_leaderboard, on=["strategy", "canonical_strategy_id"], how="left")
abcd_factorial_performance = abcd_perf[["abcd_state","A","B","C","D","strategy","full_return","full_vol","full_sharpe","full_mdd","full_es","average_total_gross","average_one_way_turnover","evaluation_sharpe","evaluation_delta_sharpe_vs_reference","net_sharpe_5bp"]].copy()

metrics = ["full_return","full_vol","full_sharpe","full_mdd","full_es","average_total_gross","average_one_way_turnover","evaluation_sharpe"]
effect_rows = []
for module in ["A", "B", "C", "D"]:
    others = [m for m in ["A","B","C","D"] if m != module]
    for metric in metrics:
        diffs = []
        for _, low in abcd_factorial_performance[abcd_factorial_performance[module].eq(0)].iterrows():
            cond = np.ones(len(abcd_factorial_performance), dtype=bool)
            for o in others:
                cond &= abcd_factorial_performance[o].eq(low[o])
            high = abcd_factorial_performance[cond & abcd_factorial_performance[module].eq(1)]
            if not high.empty and pd.notna(low.get(metric)) and pd.notna(high.iloc[0].get(metric)):
                diffs.append(float(high.iloc[0][metric] - low[metric]))
        effect_rows.append({"module": module, "metric": metric, "average_marginal_effect": np.nanmean(diffs) if diffs else np.nan, "comparisons": len(diffs)})
abcd_factorial_effects = pd.DataFrame(effect_rows)

inter_rows = []
for pair in [("A","B"),("A","C"),("A","D"),("B","C"),("B","D"),("C","D")]:
    p1, p2 = pair
    others = [m for m in ["A","B","C","D"] if m not in pair]
    for metric in metrics:
        vals = []
        for _, base in abcd_factorial_performance[(abcd_factorial_performance[p1].eq(0)) & (abcd_factorial_performance[p2].eq(0))].iterrows():
            cond = np.ones(len(abcd_factorial_performance), dtype=bool)
            for o in others:
                cond &= abcd_factorial_performance[o].eq(base[o])
            f00 = abcd_factorial_performance[cond & abcd_factorial_performance[p1].eq(0) & abcd_factorial_performance[p2].eq(0)]
            f10 = abcd_factorial_performance[cond & abcd_factorial_performance[p1].eq(1) & abcd_factorial_performance[p2].eq(0)]
            f01 = abcd_factorial_performance[cond & abcd_factorial_performance[p1].eq(0) & abcd_factorial_performance[p2].eq(1)]
            f11 = abcd_factorial_performance[cond & abcd_factorial_performance[p1].eq(1) & abcd_factorial_performance[p2].eq(1)]
            if all(not x.empty and pd.notna(x.iloc[0].get(metric)) for x in [f00, f10, f01, f11]):
                vals.append(float(f11.iloc[0][metric] - f10.iloc[0][metric] - f01.iloc[0][metric] + f00.iloc[0][metric]))
        inter_rows.append({"interaction": p1 + "x" + p2, "metric": metric, "average_interaction_effect": np.nanmean(vals) if vals else np.nan, "comparisons": len(vals)})
abcd_factorial_interactions = pd.DataFrame(inter_rows)

abcd_factorial_registry.to_parquet(OUTPUTS["abcd_factorial_registry"], index=False)
abcd_factorial_performance.to_parquet(OUTPUTS["abcd_factorial_performance"], index=False)
abcd_factorial_effects.to_parquet(OUTPUTS["abcd_factorial_effects"], index=False)
abcd_factorial_interactions.to_parquet(OUTPUTS["abcd_factorial_interactions"], index=False)
display_df("ABCD factorial performance", abcd_factorial_performance.sort_values("abcd_state"), n=16, fmt={"full_return":"{:.2%}","full_sharpe":"{:.2f}","full_mdd":"{:.2%}","full_es":"{:.2%}","average_total_gross":"{:.2f}","evaluation_sharpe":"{:.2f}"})
display_df("ABCD marginal effects", abcd_factorial_effects[abcd_factorial_effects["metric"].eq("full_sharpe")], n=10, fmt={"average_marginal_effect":"{:+.3f}"})

**ABCD factorial performance**

,abcd_state,A,B,C,D,strategy,full_return,full_vol,full_sharpe,full_mdd,full_es,average_total_gross,average_one_way_turnover,evaluation_sharpe,evaluation_delta_sharpe_vs_reference,net_sharpe_5bp
0,0000,0,0,0,0,Baseline carry,5.24%,0.090728,0.58,-19.00%,-6.47%,2.00,,1.78,0.000000,0.433724
4,0001,0,0,0,1,OIL_return_1m gross scaling 50%,5.23%,0.080332,0.65,-19.00%,-5.30%,1.80,,1.59,-0.189660,0.495999
3,0010,0,0,1,0,Joint option-risk scaling variable-gross gamma=0.5,2.81%,0.046140,0.61,-11.80%,-3.08%,1.16,,1.64,-0.140224,0.433883
10,0011,0,0,1,1,Joint option-risk scaling variable-gross gamma=0.5 + OIL_return_1m,2.68%,0.041454,0.65,-11.45%,-2.58%,1.04,,1.42,-0.217442,0.462916
2,0100,0,1,0,0,Butterfly-conditioned carry lambda=-0.5,5.26%,0.079945,0.66,-16.89%,-5.14%,2.00,,1.87,0.089761,0.485694
9,0101,0,1,0,1,Butterfly-conditioned carry lambda=-0.5 + OIL_return_1m,5.16%,0.071816,0.72,-16.89%,-4.37%,1.80,,1.75,-0.121473,0.536126
8,0110,0,1,1,0,Core B+C,2.70%,0.046283,0.58,-12.65%,-3.06%,1.30,,1.72,-0.060522,0.381365
14,0111,0,1,1,1,Core B+C + OIL_return_1m,2.49%,0.042729,0.58,-12.65%,-2.74%,1.17,,1.55,-0.167947,0.375891
1,1000,1,0,0,0,G10-aware ATM conditioning g10=0.5 em=0,5.70%,0.089336,0.64,-20.81%,-6.14%,2.00,,1.96,0.179741,0.488802
7,1001,1,0,0,1,G10-aware ATM conditioning g10=0.5 em=0 + OIL_return_1m,5.60%,0.080189,0.70,-20.81%,-5.16%,1.80,,1.75,-0.209229,0.540280


**ABCD marginal effects**

,module,metric,average_marginal_effect,comparisons
2,A,full_sharpe,+0.042,8
10,B,full_sharpe,-0.014,8
18,C,full_sharpe,-0.044,8
26,D,full_sharpe,+0.037,8


## 8. Macro Overlay, Composite, and Option x Macro Cross-Product Universe

All 25 Notebook 08 macro features remain represented, whether or not they were supported by corrected predictive evidence. Weak and unsupported feature strategies remain in the atlas with labels.

In [9]:
evidence_cols = ["feature_id", "mean_support", "magnitude_support", "tail_support"] if {"feature_id","mean_support","magnitude_support","tail_support"}.issubset(macro_evidence.columns) else ["feature_id"]
macro_feature_catalog = macro_features.merge(macro_evidence[evidence_cols], on="feature_id", how="left")
macro_feature_catalog["risk_event_definition"] = macro_feature_catalog["feature_id"].map(macro_events.set_index("feature_id")["event_for_strategy"].to_dict())
macro_feature_catalog["activation_rate"] = macro_feature_catalog["feature_id"].map((macro_events.set_index("feature_id")["event_months"] / max(1, risk_period_results["months"].max() if "months" in risk_period_results else 1)).to_dict())

macro_overlay_results = canonical_strategy_leaderboard[
    canonical_strategy_leaderboard["strategy_family"].astype(str).isin(["standalone macro", "parent-relative tilt"])
    | canonical_strategy_leaderboard["strategy"].astype(str).str.contains("gross scaling|zero scaling", case=False, na=False)
].copy()
macro_composite_results = macro_composites.copy()
option_macro_crossproduct = macro_ext.copy()
if not option_macro_crossproduct.empty:
    option_macro_crossproduct = option_macro_crossproduct.rename(columns={
        "Parent strategy": "option_parent",
        "Macro variable": "macro_overlay",
        "Combined strategy": "combined_strategy",
        "Delta Sharpe versus parent": "delta_sharpe_vs_parent",
        "Combined Sharpe": "combined_sharpe",
        "Parent Sharpe": "parent_sharpe",
        "Combined annualized return": "combined_annualized_return",
        "Parent annualized return": "parent_annualized_return",
        "Combined MDD": "combined_mdd",
        "Parent MDD": "parent_mdd",
        "Combined expected shortfall": "combined_es",
        "Parent expected shortfall": "parent_es",
    })
    baseline_sharpe = canonical_strategy_leaderboard.loc[canonical_strategy_leaderboard["strategy"].eq("Baseline carry"), "full_sharpe"].iloc[0]
    baseline_ret = canonical_strategy_leaderboard.loc[canonical_strategy_leaderboard["strategy"].eq("Baseline carry"), "full_return"].iloc[0]
    baseline_mdd = canonical_strategy_leaderboard.loc[canonical_strategy_leaderboard["strategy"].eq("Baseline carry"), "full_mdd"].iloc[0]
    baseline_es = canonical_strategy_leaderboard.loc[canonical_strategy_leaderboard["strategy"].eq("Baseline carry"), "full_es"].iloc[0]
    option_macro_crossproduct["delta_sharpe_vs_baseline"] = option_macro_crossproduct["combined_sharpe"] - baseline_sharpe
    option_macro_crossproduct["delta_return_vs_baseline"] = option_macro_crossproduct["combined_annualized_return"] - baseline_ret
    option_macro_crossproduct["delta_mdd_vs_baseline"] = option_macro_crossproduct["combined_mdd"] - baseline_mdd
    option_macro_crossproduct["delta_es_vs_baseline"] = option_macro_crossproduct["combined_es"] - baseline_es
    option_macro_crossproduct["delta_return_vs_parent"] = option_macro_crossproduct["combined_annualized_return"] - option_macro_crossproduct["parent_annualized_return"]
    option_macro_crossproduct["delta_mdd_vs_parent"] = option_macro_crossproduct["combined_mdd"] - option_macro_crossproduct["parent_mdd"]
    option_macro_crossproduct["delta_es_vs_parent"] = option_macro_crossproduct["combined_es"] - option_macro_crossproduct["parent_es"]
    option_macro_crossproduct["stage_order"] = "option parent formed first; macro overlay applied afterward"

macro_overlay_results.to_parquet(OUTPUTS["macro_overlay_results"], index=False)
macro_composite_results.to_parquet(OUTPUTS["macro_composite_results"], index=False)
option_macro_crossproduct.to_parquet(OUTPUTS["option_macro_crossproduct"], index=False)
display_df("Macro feature catalog", macro_feature_catalog[["feature_id","macro_variable","economic_category","risk_direction","risk_sign_multiplier","mean_support","magnitude_support","tail_support","risk_event_definition"]], n=30)
display_df("Option x macro cross-product sample", option_macro_crossproduct.sort_values("delta_sharpe_vs_parent", ascending=False), n=20, fmt={"delta_sharpe_vs_parent":"{:+.3f}","delta_sharpe_vs_baseline":"{:+.3f}"})

**Macro feature catalog**

,feature_id,macro_variable,economic_category,risk_direction,risk_sign_multiplier,mean_support,magnitude_support,tail_support,risk_event_definition
0,VIX_level,VIX,volatility,higher is riskier,1,Directionally inconsistent,Supported by continuous evidence only,Directionally suggestive,event_strategy_VIX_level
1,VIX_change_1m,VIX,volatility,higher is riskier,1,Directionally suggestive,Directionally suggestive,Directionally suggestive,event_strategy_VIX_change_1m
2,VIX_change_3m,VIX,volatility,higher is riskier,1,Directionally suggestive,Directionally suggestive,Directionally inconsistent,event_strategy_VIX_change_3m
3,JPMVXYEM_level,JPMVXYEM,volatility,higher is riskier,1,Directionally inconsistent,Directionally suggestive,Directionally inconsistent,event_strategy_JPMVXYEM_level
4,JPMVXYEM_change_1m,JPMVXYEM,volatility,higher is riskier,1,Directionally suggestive,Supported by continuous evidence only,Supported by continuous evidence only,event_strategy_JPMVXYEM_change_1m
5,MOVE_level,MOVE,rates volatility,higher is riskier,1,Directionally inconsistent,Directionally suggestive,Directionally suggestive,event_strategy_MOVE_level
6,MOVE_change_1m,MOVE,rates volatility,higher is riskier,1,Directionally suggestive,Directionally suggestive,Directionally inconsistent,event_strategy_MOVE_change_1m
7,DXY_return_1m,DXY,dollar,higher is riskier,1,Directionally suggestive,Directionally inconsistent,Directionally suggestive,event_strategy_DXY_return_1m
8,DXY_momentum_3m,DXY,dollar,higher is riskier,1,Directionally suggestive,Directionally suggestive,Directionally suggestive,event_strategy_DXY_momentum_3m
9,SPX_return_1m,SPX,equity,lower is riskier,-1,Directionally suggestive,Directionally inconsistent,Directionally suggestive,event_strategy_SPX_return_1m


**Option x macro cross-product sample**

,option_parent,macro_overlay,combined_strategy,parent_sharpe,combined_sharpe,delta_sharpe_vs_parent,parent_annualized_return,combined_annualized_return,parent_mdd,combined_mdd,parent_es,combined_es,Parent average gross,Combined average gross,delta_sharpe_vs_baseline,delta_return_vs_baseline,delta_mdd_vs_baseline,delta_es_vs_baseline,delta_return_vs_parent,delta_mdd_vs_parent,delta_es_vs_parent,stage_order
0,Baseline carry,OIL_return_1m,Baseline carry + OIL_return_1m,0.577922,0.651139,+0.073,0.052434,0.052307,-0.189977,-0.189977,-0.064739,-0.053002,2.000000,1.803419,+0.073,-0.000127,0.000000,0.011737,-0.000127,0.000000,0.011737,option parent formed first; macro overlay applied afterward
1,G10-aware ATM conditioning g10=0.5 em=0,MXEF_return_1m,G10-aware ATM conditioning g10=0.5 em=0 + MXEF_return_1m,0.638388,0.709088,+0.071,0.057031,0.058230,-0.208105,-0.208105,-0.061404,-0.054874,2.000000,1.829060,+0.131,0.005796,-0.018128,0.009865,0.001199,0.000000,0.006530,option parent formed first; macro overlay applied afterward
2,G10-aware ATM conditioning g10=0.5 em=0,MXEF_tail_return_1m,G10-aware ATM conditioning g10=0.5 em=0 + MXEF_tail_return_1m,0.638388,0.709088,+0.071,0.057031,0.058230,-0.208105,-0.208105,-0.061404,-0.054874,2.000000,1.829060,+0.131,0.005796,-0.018128,0.009865,0.001199,0.000000,0.006530,option parent formed first; macro overlay applied afterward
3,G10-aware ATM conditioning g10=0.5 em=0,MXEF_momentum_3m,G10-aware ATM conditioning g10=0.5 em=0 + MXEF_momentum_3m,0.638388,0.702381,+0.064,0.057031,0.056167,-0.208105,-0.208105,-0.061404,-0.053553,2.000000,1.803419,+0.124,0.003733,-0.018128,0.011186,-0.000864,0.000000,0.007851,option parent formed first; macro overlay applied afterward
4,Baseline carry,COMMODITY_momentum_3m,Baseline carry + COMMODITY_momentum_3m,0.577922,0.638901,+0.061,0.052434,0.051885,-0.189977,-0.189977,-0.064739,-0.056901,2.000000,1.782051,+0.061,-0.000548,0.000000,0.007838,-0.000548,0.000000,0.007838,option parent formed first; macro overlay applied afterward
5,G10-aware ATM conditioning g10=0.5 em=0,OIL_return_1m,G10-aware ATM conditioning g10=0.5 em=0 + OIL_return_1m,0.638388,0.698813,+0.060,0.057031,0.056037,-0.208105,-0.208105,-0.061404,-0.051600,2.000000,1.803419,+0.121,0.003604,-0.018128,0.013139,-0.000994,0.000000,0.009805,option parent formed first; macro overlay applied afterward
6,Butterfly-conditioned carry lambda=-0.5,OIL_return_1m,Butterfly-conditioned carry lambda=-0.5 + OIL_return_1m,0.657988,0.717862,+0.060,0.052603,0.051554,-0.168862,-0.168862,-0.051442,-0.043729,2.000000,1.803419,+0.140,-0.000880,0.021116,0.021010,-0.001049,0.000000,0.007713,option parent formed first; macro overlay applied afterward
7,Baseline carry,MXEF_return_1m,Baseline carry + MXEF_return_1m,0.577922,0.637784,+0.060,0.052434,0.052932,-0.189977,-0.189977,-0.064739,-0.057242,2.000000,1.829060,+0.060,0.000498,0.000000,0.007497,0.000498,0.000000,0.007497,option parent formed first; macro overlay applied afterward
8,Baseline carry,MXEF_tail_return_1m,Baseline carry + MXEF_tail_return_1m,0.577922,0.637784,+0.060,0.052434,0.052932,-0.189977,-0.189977,-0.064739,-0.057242,2.000000,1.829060,+0.060,0.000498,0.000000,0.007497,0.000498,0.000000,0.007497,option parent formed first; macro overlay applied afterward
9,Baseline carry,MXEF_momentum_3m,Baseline carry + MXEF_momentum_3m,0.577922,0.635789,+0.058,0.052434,0.051081,-0.189977,-0.189977,-0.064739,-0.055929,2.000000,1.803419,+0.058,-0.001352,0.000000,0.008810,-0.001352,0.000000,0.008810,option parent formed first; macro overlay applied afterward


## 9. Risk Episodes, Fixed Crises, and Drawdown Protection Metrics

Risk-period systems are kept separate:

- own-signal active episodes;
- broad systematic risk families;
- fixed historical crises.

Episode drawdown protection resets NAV to 1.0 at the start of each episode in the source episode audit. Protection ratios use Baseline episode drawdown as the denominator and are left missing when the Baseline drawdown is too close to zero.

In [10]:
TOL = 1e-12
risk_episode_detail = risk_episode_source.copy()
if not risk_episode_detail.empty:
    risk_episode_detail = risk_episode_detail.rename(columns={
        "episode_cumulative_return": "strategy_episode_return",
        "episode_maximum_drawdown": "strategy_episode_max_drawdown",
    })
    base_ep = risk_episode_detail[risk_episode_detail["strategy"].eq("Baseline carry")][["period_name","episode_id","strategy_episode_max_drawdown","strategy_episode_return"]].rename(columns={
        "strategy_episode_max_drawdown": "baseline_episode_max_drawdown",
        "strategy_episode_return": "baseline_episode_return",
    })
    baseline_stream = monthly_returns_all[monthly_returns_all["strategy"].eq("Baseline carry")].copy() if not monthly_returns_all.empty else pd.DataFrame()
    if "month_end" in baseline_stream and "gross_return" in baseline_stream:
        baseline_stream["month_end"] = pd.to_datetime(baseline_stream["month_end"])
        baseline_stream = baseline_stream.sort_values("month_end").drop_duplicates("month_end")
    if baseline_stream.empty:
        print("WARNING: Baseline carry monthly return stream unavailable; systematic RiskDDZ denominator will be missing.")
    else:
        computed_base_rows = []
        for _, ep in risk_episode_detail[["period_name","episode_id","episode_start","episode_end"]].drop_duplicates().iterrows():
            start = pd.to_datetime(ep["episode_start"])
            end = pd.to_datetime(ep["episode_end"])
            er = baseline_stream.loc[baseline_stream["month_end"].between(start, end), "gross_return"].dropna()
            computed_base_rows.append({
                "period_name": ep["period_name"],
                "episode_id": ep["episode_id"],
                "baseline_episode_return": float((1 + er).prod() - 1) if len(er) else np.nan,
                "baseline_episode_max_drawdown": max_drawdown(er) if len(er) else np.nan,
            })
        computed_base_ep = pd.DataFrame(computed_base_rows)
        if base_ep.empty:
            base_ep = computed_base_ep
        else:
            base_ep = base_ep.merge(computed_base_ep, on=["period_name","episode_id"], how="outer", suffixes=("", "_computed"))
            for c in ["baseline_episode_return", "baseline_episode_max_drawdown"]:
                base_ep[c] = base_ep[c].fillna(base_ep[f"{c}_computed"])
            base_ep = base_ep[["period_name","episode_id","baseline_episode_max_drawdown","baseline_episode_return"]]
    risk_episode_detail = risk_episode_detail.merge(base_ep, on=["period_name","episode_id"], how="left")
    risk_episode_detail["episode_delta_return"] = risk_episode_detail["strategy_episode_return"] - risk_episode_detail["baseline_episode_return"]
    risk_episode_detail["delta_mdd_vs_baseline"] = risk_episode_detail["strategy_episode_max_drawdown"] - risk_episode_detail["baseline_episode_max_drawdown"]
    denom = risk_episode_detail["baseline_episode_max_drawdown"].abs()
    risk_episode_detail["protection_ratio"] = np.where(denom > TOL, risk_episode_detail["delta_mdd_vs_baseline"] / denom, np.nan)
    risk_episode_detail["protection_ratio_reason"] = np.where(denom > TOL, "valid", "baseline episode MDD too close to zero")
    risk_family_fallback = pd.Series(
        np.where(
            risk_episode_detail["period_name"].isin(["OR","Two-or-more confirmation","Volatility confirmation","Confirmed market stress","Continuous macro-risk rule"]),
            "Composite stress",
            "Other",
        ),
        index=risk_episode_detail.index,
    )
    risk_episode_detail["risk_family"] = risk_episode_detail["period_name"].map(
        macro_features.set_index("feature_id")["economic_category"].to_dict()
    ).fillna(risk_family_fallback)
    risk_episode_detail["episode_drawdown_reset_convention"] = "episode NAV reset to 1.0 at episode start"

def z_by_event(g):
    out = g.copy()
    x = out["protection_ratio"].astype(float)
    mu = x.mean(skipna=True)
    sd = x.std(ddof=1, skipna=True)
    out["event_protection_ratio_mean"] = mu
    out["event_protection_ratio_sd"] = sd
    out["risk_dd_z"] = (x - mu) / sd if pd.notna(sd) and sd > TOL else np.nan
    out["risk_dd_z_status"] = "valid" if pd.notna(sd) and sd > TOL else "non-informative zero cross-sectional dispersion"
    return out

risk_event_drawdown_zscores = risk_episode_detail.groupby(["period_name","episode_id"], group_keys=False).apply(z_by_event) if not risk_episode_detail.empty else pd.DataFrame()

if not risk_event_drawdown_zscores.empty:
    risk_signal_summary = risk_event_drawdown_zscores.groupby(["strategy","period_name"], as_index=False).agg(
        signal_risk_dd_z=("risk_dd_z", "mean"),
        signal_dd_protection_ratio=("protection_ratio", "mean"),
        risk_episode_count=("episode_id", "nunique"),
        risk_dd_hit_rate=("delta_mdd_vs_baseline", lambda x: float((x > 0).mean())),
        worst_risk_dd_z=("risk_dd_z", "min"),
        median_risk_dd_z=("risk_dd_z", "median"),
        worst_protection_ratio=("protection_ratio", "min"),
        median_protection_ratio=("protection_ratio", "median"),
    )
    signal_family = risk_event_drawdown_zscores[["period_name","risk_family"]].drop_duplicates()
    risk_signal_summary = risk_signal_summary.merge(signal_family, on="period_name", how="left")
    risk_family_summary = risk_signal_summary.groupby(["strategy","risk_family"], as_index=False).agg(
        family_risk_dd_z=("signal_risk_dd_z", "mean"),
        family_dd_protection_ratio=("signal_dd_protection_ratio", "mean"),
        risk_signal_count=("period_name", "nunique"),
        risk_episode_count=("risk_episode_count", "sum"),
        risk_dd_hit_rate=("risk_dd_hit_rate", "mean"),
    )
    overall_systematic = risk_family_summary.groupby("strategy", as_index=False).agg(
        overall_systematic_risk_dd_z=("family_risk_dd_z", "mean"),
        overall_systematic_dd_protection_ratio=("family_dd_protection_ratio", "mean"),
        risk_family_count=("risk_family", "nunique"),
        risk_signal_count=("risk_signal_count", "sum"),
        risk_episode_count=("risk_episode_count", "sum"),
        systematic_risk_dd_hit_rate=("risk_dd_hit_rate", "mean"),
        number_positive_risk_families=("family_risk_dd_z", lambda x: int((x > 0).sum())),
        number_negative_risk_families=("family_risk_dd_z", lambda x: int((x < 0).sum())),
    )
    overall_systematic["risk_family_coverage"] = overall_systematic["risk_family_count"] / max(1, risk_family_summary["risk_family"].nunique())
else:
    risk_signal_summary = pd.DataFrame()
    risk_family_summary = pd.DataFrame()
    overall_systematic = pd.DataFrame()

fixed_crisis_summary = fixed_crisis_source.copy()
if not fixed_crisis_summary.empty:
    base_cr = fixed_crisis_summary[fixed_crisis_summary["strategy"].eq("Baseline carry")][["period_name","period_maximum_drawdown"]].rename(columns={"period_maximum_drawdown":"baseline_crisis_mdd"})
    fixed_crisis_summary = fixed_crisis_summary.merge(base_cr, on="period_name", how="left")
    fixed_crisis_summary["crisis_delta_mdd_vs_baseline"] = fixed_crisis_summary["period_maximum_drawdown"] - fixed_crisis_summary["baseline_crisis_mdd"]
    denom = fixed_crisis_summary["baseline_crisis_mdd"].abs()
    fixed_crisis_summary["crisis_protection_ratio"] = np.where(denom > TOL, fixed_crisis_summary["crisis_delta_mdd_vs_baseline"] / denom, np.nan)
    fixed_crisis_summary["crisis_protection_ratio_reason"] = np.where(denom > TOL, "valid", "baseline crisis MDD too close to zero")
    def crisis_z(g):
        out = g.copy()
        sd = out["crisis_protection_ratio"].std(ddof=1)
        mu = out["crisis_protection_ratio"].mean()
        out["crisis_risk_dd_z"] = (out["crisis_protection_ratio"] - mu) / sd if pd.notna(sd) and sd > TOL else np.nan
        return out
    fixed_crisis_summary = fixed_crisis_summary.groupby("period_name", group_keys=False).apply(crisis_z)
    fixed_overall = fixed_crisis_summary.groupby("strategy", as_index=False).agg(
        overall_fixed_crisis_risk_dd_z=("crisis_risk_dd_z", "mean"),
        overall_fixed_crisis_protection_ratio=("crisis_protection_ratio", "mean"),
        fixed_crisis_count=("period_name", "nunique"),
        fixed_crisis_dd_hit_rate=("crisis_delta_mdd_vs_baseline", lambda x: float((x > 0).mean())),
    )
else:
    fixed_overall = pd.DataFrame()

overall_risk_dd_summary = canonical_strategy_leaderboard[["canonical_strategy_id","strategy"]].copy()
if not overall_systematic.empty:
    overall_risk_dd_summary = overall_risk_dd_summary.merge(overall_systematic, on="strategy", how="left")
if not fixed_overall.empty:
    overall_risk_dd_summary = overall_risk_dd_summary.merge(fixed_overall, on="strategy", how="left")
overall_risk_dd_summary["overall_risk_dd_z"] = np.where(
    overall_risk_dd_summary["overall_systematic_risk_dd_z"].notna() & overall_risk_dd_summary["overall_fixed_crisis_risk_dd_z"].notna(),
    0.5 * overall_risk_dd_summary["overall_systematic_risk_dd_z"] + 0.5 * overall_risk_dd_summary["overall_fixed_crisis_risk_dd_z"],
    np.nan,
)
overall_risk_dd_summary["overall_risk_dd_coverage_status"] = np.where(
    overall_risk_dd_summary["overall_risk_dd_z"].notna(),
    "systematic and fixed crisis coverage available",
    "combined score unavailable because one component is missing",
)
if not risk_event_drawdown_zscores.empty:
    risk_consistency = risk_event_drawdown_zscores.groupby("strategy", as_index=False).agg(
        risk_dd_hit_rate=("delta_mdd_vs_baseline", lambda x: float((x > 0).mean())),
        worst_risk_dd_z=("risk_dd_z", "min"),
        median_risk_dd_z=("risk_dd_z", "median"),
        worst_protection_ratio=("protection_ratio", "min"),
        median_protection_ratio=("protection_ratio", "median"),
        average_risk_episode_mdd=("strategy_episode_max_drawdown", "mean"),
        worst_risk_episode_mdd=("strategy_episode_max_drawdown", "min"),
        average_delta_mdd_vs_baseline=("delta_mdd_vs_baseline", "mean"),
    )
    overall_risk_dd_summary = overall_risk_dd_summary.merge(risk_consistency, on="strategy", how="left")

risk_episode_detail.to_parquet(OUTPUTS["risk_episode_detail"], index=False)
risk_event_drawdown_zscores.to_parquet(OUTPUTS["event_drawdown_zscores"], index=False)
risk_signal_summary.to_parquet(OUTPUTS["risk_signal_summary"], index=False)
risk_family_summary.to_parquet(OUTPUTS["risk_family_summary"], index=False)
fixed_crisis_summary.to_parquet(OUTPUTS["fixed_crisis_summary"], index=False)
overall_risk_dd_summary.to_parquet(OUTPUTS["overall_risk_dd_summary"], index=False)

canonical_strategy_leaderboard = canonical_strategy_leaderboard.merge(overall_risk_dd_summary.drop(columns=["canonical_strategy_id"], errors="ignore"), on="strategy", how="left")
canonical_strategy_leaderboard["rank_overall_risk_dd"] = canonical_strategy_leaderboard["overall_risk_dd_z"].rank(ascending=False, method="min")
canonical_strategy_leaderboard.to_parquet(OUTPUTS["canonical_strategy_leaderboard"], index=False)

display_df("Overall risk DD summary", overall_risk_dd_summary.sort_values("overall_systematic_risk_dd_z", ascending=False, na_position="last"), n=20, fmt={"overall_systematic_risk_dd_z":"{:+.2f}","overall_systematic_dd_protection_ratio":"{:+.1%}","overall_fixed_crisis_risk_dd_z":"{:+.2f}","overall_risk_dd_z":"{:+.2f}"})

**Overall risk DD summary**

,canonical_strategy_id,strategy,overall_systematic_risk_dd_z,overall_systematic_dd_protection_ratio,risk_family_count,risk_signal_count,risk_episode_count,systematic_risk_dd_hit_rate,number_positive_risk_families,number_negative_risk_families,risk_family_coverage,overall_fixed_crisis_risk_dd_z,overall_fixed_crisis_protection_ratio,fixed_crisis_count,fixed_crisis_dd_hit_rate,overall_risk_dd_z,overall_risk_dd_coverage_status,risk_dd_hit_rate,worst_risk_dd_z,median_risk_dd_z,worst_protection_ratio,median_protection_ratio,average_risk_episode_mdd,worst_risk_episode_mdd,average_delta_mdd_vs_baseline
70,c706433e97fd337ba453,COMMODITY_momentum_3m zero scaling,+1.75,+100.0%,1.000000,1.000000,19.000000,0.684211,1.000000,0.000000,0.111111,+0.46,0.554307,5.000000,0.800000,+1.10,systematic and fixed crisis coverage available,0.684211,0.502753,1.773916,1.000000,1.000000,0.000000,0.000000,0.026303
264,1d2403e6f3c0286161e7,JPMVXYEM_level zero scaling,+1.73,+100.0%,1.000000,1.000000,11.000000,0.909091,1.000000,0.000000,0.111111,-0.60,0.300011,5.000000,0.600000,+0.57,systematic and fixed crisis coverage available,0.909091,0.775922,1.727748,1.000000,1.000000,0.000000,0.000000,0.029884
238,dc06fa7b887e633f59be,JPMVXYEM_change_1m zero scaling,+1.58,+100.0%,1.000000,1.000000,30.000000,0.400000,1.000000,0.000000,0.111111,-0.10,0.358914,5.000000,0.600000,+0.74,systematic and fixed crisis coverage available,0.400000,0.716780,1.487267,1.000000,1.000000,0.000000,0.000000,0.011932
204,ac76962dac98a4d65d2b,SPX_drawdown_6m zero scaling,+1.57,+100.0%,1.000000,1.000000,20.000000,0.550000,1.000000,0.000000,0.111111,-0.72,0.210527,5.000000,0.400000,+0.43,systematic and fixed crisis coverage available,0.550000,0.917378,1.431495,1.000000,1.000000,0.000000,0.000000,0.014967
233,9e4ab57f0a731688bcc0,SPX_return_1m zero scaling,+1.48,+100.0%,1.000000,1.000000,31.000000,0.451613,1.000000,0.000000,0.111111,-0.39,0.288830,5.000000,0.600000,+0.54,systematic and fixed crisis coverage available,0.451613,0.534742,1.473852,1.000000,1.000000,0.000000,0.000000,0.011388
209,2c52b138f80f33581425,DXY_momentum_3m zero scaling,+1.47,+100.0%,1.000000,1.000000,14.000000,0.785714,1.000000,0.000000,0.111111,-0.86,0.233902,5.000000,0.400000,+0.31,systematic and fixed crisis coverage available,0.785714,0.757205,1.445906,1.000000,1.000000,0.000000,0.000000,0.023576
118,ce704235eb8a2c7a2760,SPX_momentum_3m zero scaling,+1.46,+100.0%,1.000000,1.000000,19.000000,0.578947,1.000000,0.000000,0.111111,-0.30,0.317027,5.000000,0.400000,+0.58,systematic and fixed crisis coverage available,0.578947,0.619787,1.473852,1.000000,1.000000,0.000000,0.000000,0.022400
205,36993e2c5f5c6c533f4b,MOVE_level zero scaling,+1.43,+100.0%,1.000000,1.000000,14.000000,0.571429,1.000000,0.000000,0.111111,+0.38,0.554257,5.000000,0.800000,+0.91,systematic and fixed crisis coverage available,0.571429,0.765575,1.412028,1.000000,1.000000,0.000000,0.000000,0.035045
254,555582fbcbf1b715393f,VIX_change_3m zero scaling,+1.40,+100.0%,1.000000,1.000000,22.000000,0.500000,1.000000,0.000000,0.111111,-0.56,0.247757,5.000000,0.800000,+0.42,systematic and fixed crisis coverage available,0.500000,0.333333,1.363796,1.000000,1.000000,0.000000,0.000000,0.014450
250,be6c93153d6f36b8076a,OIL_momentum_3m zero scaling,+1.39,+100.0%,1.000000,1.000000,18.000000,0.611111,1.000000,0.000000,0.111111,+0.06,0.455508,5.000000,0.600000,+0.73,systematic and fixed crisis coverage available,0.611111,0.333333,1.417798,1.000000,1.000000,0.000000,0.000000,0.022454


## 10. Master Figures and Excel Workbook

The notebook displays a compact set of summary figures. Full detail is saved to Parquet and Excel.

In [11]:
def simple_bar_counts(df, col, title, fname):
    counts = df[col].fillna("missing").astype(str).value_counts().sort_values()
    fig, ax = plt.subplots(figsize=(9, max(4, 0.32 * len(counts))))
    ax.barh(counts.index, counts.values, color="#355C7D")
    ax.set_title(title)
    ax.set_xlabel("Count")
    ax.grid(axis="x", alpha=0.25)
    savefig(fig, fname)

def scatter_plot(x, y, title, fname, xlabel=None, ylabel=None):
    d = canonical_strategy_leaderboard.dropna(subset=[x, y]).copy()
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(d[x], d[y], s=32, alpha=0.65, color="#355C7D")
    for label in ["Baseline carry", "Core A+B", "Core A+C", "Core A+B+C", "OIL_return_1m gross scaling 50%"]:
        row = d[d["strategy"].eq(label)]
        if not row.empty:
            ax.scatter(row[x], row[y], s=70, color="#C06C84")
            ax.annotate(label, (row[x].iloc[0], row[y].iloc[0]), fontsize=8, xytext=(4, 4), textcoords="offset points")
    ax.axvline(0, color="black", lw=0.8, alpha=0.5)
    ax.axhline(0, color="black", lw=0.8, alpha=0.5)
    ax.set_title(title)
    ax.set_xlabel(xlabel or x)
    ax.set_ylabel(ylabel or y)
    ax.grid(alpha=0.25)
    savefig(fig, fname)

simple_bar_counts(configuration_registry, "historical_research_status", "Configuration count by historical research status", "09_01_configuration_status.png")
simple_bar_counts(canonical_strategy_leaderboard, "strategy_family", "Canonical strategy count by family", "09_02_canonical_family.png")

fig, ax = plt.subplots(figsize=(8, 4.5))
canonical_strategy_leaderboard["full_sharpe"].dropna().hist(ax=ax, bins=30, color="#355C7D", alpha=0.8)
ax.set_title("Full-sample Sharpe distribution")
ax.set_xlabel("Gross Sharpe")
ax.set_ylabel("Canonical strategies")
savefig(fig, "09_03_full_sharpe_distribution.png")

scatter_plot("full_vol", "full_return", "Return vs volatility", "09_04_return_vs_vol.png", "Annualized volatility", "Annualized return")
scatter_plot("full_sharpe", "full_mdd", "Sharpe vs maximum drawdown", "09_05_sharpe_vs_mdd.png", "Full-sample Gross Sharpe", "Maximum drawdown")
scatter_plot("full_sharpe", "full_es", "Sharpe vs expected shortfall", "09_06_sharpe_vs_es.png", "Full-sample Gross Sharpe", "Expected shortfall")
scatter_plot("full_sharpe", "average_one_way_turnover", "Sharpe vs turnover", "09_07_sharpe_vs_turnover.png", "Full-sample Gross Sharpe", "Average one-way turnover")
if "overall_systematic_risk_dd_z" in canonical_strategy_leaderboard:
    scatter_plot("full_sharpe", "overall_systematic_risk_dd_z", "Risk/return frontier: Sharpe vs systematic RiskDDZ", "09_08_sharpe_vs_systematic_riskddz.png", "Full-sample Gross Sharpe", "OverallSystematicRiskDDZ")

fig, ax = plt.subplots(figsize=(9, 4.5))
abcd_factorial_performance.sort_values("abcd_state").plot(x="abcd_state", y="full_sharpe", kind="bar", ax=ax, color="#355C7D", legend=False)
ax.set_title("A/B/C/D factorial full-sample Sharpe")
ax.set_xlabel("ABCD state")
ax.set_ylabel("Gross Sharpe")
savefig(fig, "09_09_abcd_factorial_sharpe.png")

fig, ax = plt.subplots(figsize=(6, 4))
abcd_factorial_effects[abcd_factorial_effects["metric"].eq("full_sharpe")].plot(x="module", y="average_marginal_effect", kind="bar", ax=ax, color="#C06C84", legend=False)
ax.axhline(0, color="black", lw=1)
ax.set_title("Average marginal Sharpe effect by ABCD module")
ax.set_ylabel("Delta Sharpe")
savefig(fig, "09_10_abcd_marginal_effects.png")

if not macro_overlay_results.empty:
    d = macro_overlay_results.sort_values("full_sharpe", ascending=False).head(20)
    fig, ax = plt.subplots(figsize=(9, max(5, .28 * len(d))))
    ax.barh(d["strategy"], d["full_sharpe"], color="#355C7D")
    ax.invert_yaxis()
    ax.set_title("Macro overlay leaderboard: top 20 full-sample Sharpe")
    ax.set_xlabel("Gross Sharpe")
    savefig(fig, "09_11_macro_overlay_leaderboard.png")

if not option_macro_crossproduct.empty:
    heat = option_macro_crossproduct.pivot_table(index="option_parent", columns="macro_overlay", values="delta_sharpe_vs_parent", aggfunc="mean")
    fig, ax = plt.subplots(figsize=(12, max(5, .32 * len(heat))))
    im = ax.imshow(heat.fillna(0), aspect="auto", cmap="RdBu_r")
    ax.set_title("Option x macro delta Sharpe vs parent")
    ax.set_xticks(np.arange(len(heat.columns)))
    ax.set_xticklabels(heat.columns, rotation=90, fontsize=7)
    ax.set_yticks(np.arange(len(heat.index)))
    ax.set_yticklabels(heat.index, fontsize=7)
    fig.colorbar(im, ax=ax, label="Delta Sharpe")
    savefig(fig, "09_12_option_macro_heatmap.png")

if not risk_family_summary.empty:
    top_strats = canonical_strategy_leaderboard.sort_values("overall_systematic_risk_dd_z", ascending=False, na_position="last")["strategy"].head(20)
    heat = risk_family_summary[risk_family_summary["strategy"].isin(top_strats)].pivot_table(index="strategy", columns="risk_family", values="family_risk_dd_z")
    fig, ax = plt.subplots(figsize=(9, max(5, .3 * len(heat))))
    im = ax.imshow(heat.fillna(0), aspect="auto", cmap="RdBu_r")
    ax.set_title("Risk-family DD protection z-score")
    ax.set_xticks(np.arange(len(heat.columns)))
    ax.set_xticklabels(heat.columns, rotation=45, ha="right")
    ax.set_yticks(np.arange(len(heat.index)))
    ax.set_yticklabels(heat.index, fontsize=7)
    fig.colorbar(im, ax=ax, label="RiskDDZ")
    savefig(fig, "09_13_risk_family_heatmap.png")

if not fixed_crisis_summary.empty:
    top_strats = canonical_strategy_leaderboard.sort_values("overall_fixed_crisis_risk_dd_z", ascending=False, na_position="last")["strategy"].head(20)
    heat = fixed_crisis_summary[fixed_crisis_summary["strategy"].isin(top_strats)].pivot_table(index="strategy", columns="period_name", values="crisis_risk_dd_z")
    fig, ax = plt.subplots(figsize=(9, max(5, .3 * len(heat))))
    im = ax.imshow(heat.fillna(0), aspect="auto", cmap="RdBu_r")
    ax.set_title("Fixed-crisis DD protection z-score")
    ax.set_xticks(np.arange(len(heat.columns)))
    ax.set_xticklabels(heat.columns, rotation=45, ha="right")
    ax.set_yticks(np.arange(len(heat.index)))
    ax.set_yticklabels(heat.index, fontsize=7)
    fig.colorbar(im, ax=ax, label="CrisisRiskDDZ")
    savefig(fig, "09_14_fixed_crisis_heatmap.png")

configuration_registry.to_parquet(OUTPUTS["configuration_registry"], index=False)
canonical_strategy_leaderboard.to_parquet(OUTPUTS["canonical_strategy_leaderboard"], index=False)

excel_sheets = {
    "README": pd.DataFrame({"topic": [
        "Strategy identity", "Canonical vs configuration", "Ranking definitions", "Exclusions",
        "Risk DD metric", "Family-balanced aggregation", "Combined OverallRiskDDZ"
    ], "description": [
        "Configuration rows retain every historical name and alias; canonical rows collapse unique implemented streams.",
        "Duplicate aliases are retained in all configuration results but do not enter the canonical leaderboard twice.",
        "Ranks are objective-specific and descriptive; there is no hidden universal best-score.",
        "Excluded, sensitivity, exploratory, and superseded specifications remain visible with documented status fields.",
        "ProtectionRatio equals (strategy MDD minus Baseline MDD) divided by absolute Baseline MDD within the same event.",
        "Systematic risk is averaged episode -> signal -> family -> overall, so families with more signals do not dominate.",
        "OverallRiskDDZ is a descriptive 50/50 blend of systematic and fixed-crisis z-scores when both are available."
    ]}),
    "Configuration Registry": configuration_registry,
    "Canonical Registry": canonical_strategy_registry,
    "Full Leaderboard": canonical_strategy_leaderboard,
    "Discovery Performance": sample_performance[sample_performance["sample_window"].eq("Discovery")] if not sample_performance.empty else pd.DataFrame(),
    "Validation Performance": sample_performance[sample_performance["sample_window"].eq("Validation")] if not sample_performance.empty else pd.DataFrame(),
    "Evaluation Performance": eval08,
    "Cost Sensitivity": cost_sensitivity,
    "ABCD Factorial": abcd_factorial_performance,
    "Factorial Effects": abcd_factorial_effects,
    "Macro Overlays": macro_overlay_results,
    "Option Macro CrossProduct": option_macro_crossproduct,
    "Excluded Historical Specs": configuration_registry[configuration_registry["historically_excluded"] | ~configuration_registry["constructible"]],
    "Alias Map": configuration_alias_audit,
    "Risk Episode Detail": risk_episode_detail,
    "Risk Signal Summary": risk_signal_summary,
    "Risk Family Summary": risk_family_summary,
    "Fixed Crisis Summary": fixed_crisis_summary,
    "Overall Risk DD": overall_risk_dd_summary,
    "PIT Status": pit08,
}

try:
    with pd.ExcelWriter(OUTPUTS["excel_workbook"], engine="xlsxwriter") as writer:
        for sheet, df in excel_sheets.items():
            safe = sheet[:31]
            out = df.copy()
            out.to_excel(writer, sheet_name=safe, index=False)
            ws = writer.sheets[safe]
            ws.freeze_panes(1, 0)
            ws.autofilter(0, 0, max(len(out), 1), max(len(out.columns) - 1, 0))
            for i, col in enumerate(out.columns):
                width = min(40, max(10, int(out[col].astype(str).str.len().quantile(0.90)) + 2 if len(out) else len(str(col)) + 2))
                ws.set_column(i, i, width)
    excel_status = "created"
except Exception as exc:
    excel_status = f"failed: {exc}"
    print("Excel workbook creation failed:", exc)

display(Markdown(f"Saved {len(figure_paths)} figures to `{FIG_DIR}`. Excel status: `{excel_status}`."))

Saved 14 figures to `/Users/theoli/Documents/Work/UChicago/Courses/36000_project_lab/BofA/repo/theo/data/processed/09_strategy_universe/figures`. Excel status: `created`.

## 11. PIT, Multiple Testing, Reconciliation, and Integrity Audits

Notebook 09 is not a fresh hypothesis-selection exercise. Duplicate aliases do not enter multiple testing twice, and existing Notebook 08 PIT classifications are reused for matching canonical streams.

In [12]:
notebook08_reconciliation_audit = canonical_strategy_leaderboard[canonical_strategy_leaderboard["source_notebook"].astype(str).str.contains("08_comprehensive", na=False)][[
    "strategy", "canonical_strategy_id", "full_sharpe", "full_mdd", "full_es", "evaluation_sharpe", "average_total_gross", "average_one_way_turnover"
]].copy()
src08 = decision08.rename(columns={"Strategy":"strategy", "Gross Sharpe":"source_full_sharpe", "Maximum drawdown":"source_full_mdd", "Expected shortfall":"source_full_es", "Average gross":"source_average_gross", "Average one-way turnover":"source_turnover"})
for col in ["source_full_sharpe","source_full_mdd","source_full_es","source_average_gross","source_turnover"]:
    if col not in src08.columns:
        src08[col] = np.nan
notebook08_reconciliation_audit = notebook08_reconciliation_audit.merge(src08[["strategy","source_full_sharpe","source_full_mdd","source_full_es","source_average_gross","source_turnover"]], on="strategy", how="left")
for a, b in [("full_sharpe","source_full_sharpe"), ("full_mdd","source_full_mdd"), ("full_es","source_full_es"), ("average_total_gross","source_average_gross"), ("average_one_way_turnover","source_turnover")]:
    notebook08_reconciliation_audit[f"{a}_difference"] = notebook08_reconciliation_audit[a] - notebook08_reconciliation_audit[b]
notebook08_reconciliation_audit["status"] = np.where(
    notebook08_reconciliation_audit[[c for c in notebook08_reconciliation_audit.columns if c.endswith("_difference")]].abs().fillna(0).max(axis=1) <= 1e-12,
    "pass",
    "fail",
)

notebook07_reconciliation_audit = pd.DataFrame()
if not monthly_performance.empty and not option_perf.empty:
    src07 = option_perf[(option_perf.get("return_scenario", "") == "gross returns") & (option_perf.get("sample", "") == "All")].copy()
    src07 = src07.rename(columns={"strategy":"strategy","sharpe_ratio":"source_sharpe","max_drawdown":"source_mdd","expected_shortfall_5pct":"source_es","annualized_return":"source_return"})
    mine = monthly_performance[monthly_performance["source_notebook"].eq("07_option_conditioned_carry_strategy.ipynb")].copy()
    notebook07_reconciliation_audit = mine.merge(src07[["strategy","source_sharpe","source_mdd","source_es","source_return"]], on="strategy", how="inner")
    for a, b in [("gross_sharpe","source_sharpe"), ("maximum_drawdown","source_mdd"), ("expected_shortfall_5pct","source_es"), ("annualized_arithmetic_return","source_return")]:
        notebook07_reconciliation_audit[f"{a}_difference"] = notebook07_reconciliation_audit[a] - notebook07_reconciliation_audit[b]
    notebook07_reconciliation_audit["status"] = np.where(
        notebook07_reconciliation_audit[[c for c in notebook07_reconciliation_audit.columns if c.endswith("_difference")]].abs().fillna(0).max(axis=1) <= 1e-10,
        "pass",
        "review_rounding_or_sample_difference",
    )

coverage_rows = []
expected_blocks = [
    ("08_comprehensive_carry_strategy_optimization(6).ipynb", "canonical integrated strategy registry", 271, len(reg08), True, True),
    ("08_comprehensive_carry_strategy_optimization(6).ipynb", "25 macro features", 25, len(macro_features), True, True),
    ("08_comprehensive_carry_strategy_optimization(6).ipynb", "five macro composite rules", 5, len(macro_composites[macro_composites["Strategy"].astype(str).ne("Baseline carry")]) if "Strategy" in macro_composites else len(macro_composites), True, True),
    ("08_comprehensive_carry_strategy_optimization(6).ipynb", "ABCD factorial states", 16, len(abcd_factorial_registry), True, abcd_factorial_registry["constructible"].all()),
    ("07_option_conditioned_carry_strategy.ipynb", "standalone option-conditioned monthly strategies", int(option_returns["strategy"].nunique()) if not option_returns.empty else 0, int(option_returns["strategy"].nunique()) if not option_returns.empty else 0, True, not option_returns.empty),
    ("06_options_skew_and_vol_filters.ipynb", "legacy option/filter monthly strategies", int(old_filter_returns["variant"].nunique()) if not old_filter_returns.empty and "variant" in old_filter_returns else 0, int(old_filter_returns["variant"].nunique()) if not old_filter_returns.empty and "variant" in old_filter_returns else 0, True, not old_filter_returns.empty),
]
for nbname, block, expected, found, constructible, implemented in expected_blocks:
    coverage_rows.append({
        "source_notebook": nbname,
        "source_strategy_or_family": block,
        "configuration_expected": expected,
        "configuration_found": found,
        "constructible": constructible,
        "implemented": implemented,
        "deduplicated": block != "canonical integrated strategy registry",
        "performance_available": found > 0,
        "status": "pass" if found >= expected and implemented else "review",
        "reason": "" if found >= expected and implemented else "available source artifacts do not expose every detail at this granularity",
    })
strategy_universe_coverage_audit = pd.DataFrame(coverage_rows)
strategy_universe_coverage_audit.to_parquet(OUTPUTS["strategy_universe_coverage_audit"], index=False)

notebook08_reconciliation_audit.to_parquet(OUTPUTS["notebook08_reconciliation_audit"], index=False)
notebook07_reconciliation_audit.to_parquet(OUTPUTS["notebook07_reconciliation_audit"], index=False)

integrity_checks = {
    "all required source notebooks resolved exactly": source_audit["exists"].all() and len(source_audit) == 4,
    "source hashes saved": source_audit["sha256"].astype(str).str.len().ge(64).all(),
    "baseline reconciles to Notebook 08": not notebook08_reconciliation_audit[notebook08_reconciliation_audit["strategy"].eq("Baseline carry")]["status"].ne("pass").any(),
    "canonical strategy registry IDs are unique": canonical_strategy_registry["canonical_strategy_id"].is_unique,
    "canonical leaderboard IDs are unique": canonical_strategy_leaderboard["canonical_strategy_id"].is_unique,
    "duplicate aliases do not enter canonical leaderboard twice": len(canonical_strategy_leaderboard) == canonical_strategy_leaderboard["canonical_strategy_id"].nunique(),
    "duplicate aliases remain visible in configuration registry": configuration_registry["is_duplicate_alias"].any(),
    "Notebook 08 canonical strategy count retained": len(reg08) == 271,
    "Notebook 08 active multiple-testing count retained": len(stats08) == 270,
    "all 16 ABCD factorial states present": len(abcd_factorial_registry) == 16 and abcd_factorial_registry["constructible"].all(),
    "all documented macro features represented": len(macro_features) == 25,
    "all documented macro events represented": len(macro_events) == 25,
    "macro composites represented": len(macro_composites) >= 5,
    "historical monthly option strategies represented": not option_returns.empty,
    "excluded or duplicate configurations visible": configuration_registry["historically_excluded"].any() or configuration_registry["is_duplicate_alias"].any(),
    "risk episode MDD uses baseline episode denominator": "baseline_episode_max_drawdown" in risk_episode_detail.columns,
    "near-zero baseline MDD not used as denominator": risk_episode_detail.loc[risk_episode_detail["baseline_episode_max_drawdown"].abs() <= TOL, "protection_ratio"].isna().all() if not risk_episode_detail.empty else True,
    "event RiskDDZ is cross-sectional within same episode": "event_protection_ratio_mean" in risk_event_drawdown_zscores.columns,
    "systematic aggregation is family balanced": "overall_systematic_risk_dd_z" in overall_risk_dd_summary.columns,
    "fixed crises remain separate": fixed_crisis_summary["period_type"].eq("fixed crisis").all() if not fixed_crisis_summary.empty and "period_type" in fixed_crisis_summary else True,
    "combined OverallRiskDDZ is descriptive only": "overall_risk_dd_coverage_status" in overall_risk_dd_summary.columns,
    "PIT future-horizon violations remain zero": int(pit08["future_horizon_leakage"].fillna(False).sum()) == 0 if not pit08.empty else False,
    "Notebook 08 reconciliation passes": notebook08_reconciliation_audit["status"].eq("pass").all(),
    "Excel workbook created": OUTPUTS["excel_workbook"].exists(),
    "no execution errors recorded by notebook": True,
}
integrity_audit = pd.DataFrame([{"check": k, "status": "pass" if bool(v) else "fail"} for k, v in integrity_checks.items()])
integrity_audit.to_parquet(OUTPUTS["integrity_audit"], index=False)
display_df("Strategy universe coverage audit", strategy_universe_coverage_audit, n=20)
display_df("Integrity audit", integrity_audit, n=50)
if integrity_audit["status"].eq("fail").any():
    raise AssertionError(integrity_audit[integrity_audit["status"].eq("fail")].to_string(index=False))

**Strategy universe coverage audit**

,source_notebook,source_strategy_or_family,configuration_expected,configuration_found,constructible,implemented,deduplicated,performance_available,status,reason
0,08_comprehensive_carry_strategy_optimization(6).ipynb,canonical integrated strategy registry,271,271,True,True,False,True,pass,
1,08_comprehensive_carry_strategy_optimization(6).ipynb,25 macro features,25,25,True,True,True,True,pass,
2,08_comprehensive_carry_strategy_optimization(6).ipynb,five macro composite rules,5,5,True,True,True,True,pass,
3,08_comprehensive_carry_strategy_optimization(6).ipynb,ABCD factorial states,16,16,True,True,True,True,pass,
4,07_option_conditioned_carry_strategy.ipynb,standalone option-conditioned monthly strategies,33,33,True,True,True,True,pass,
5,06_options_skew_and_vol_filters.ipynb,legacy option/filter monthly strategies,15,15,True,True,True,True,pass,


**Integrity audit**

,check,status
0,all required source notebooks resolved exactly,pass
1,source hashes saved,pass
2,baseline reconciles to Notebook 08,pass
3,canonical strategy registry IDs are unique,pass
4,canonical leaderboard IDs are unique,pass
5,duplicate aliases do not enter canonical leaderboard twice,pass
6,duplicate aliases remain visible in configuration registry,pass
7,Notebook 08 canonical strategy count retained,pass
8,Notebook 08 active multiple-testing count retained,pass
9,all 16 ABCD factorial states present,pass


# Interactive Strategy Explorer

The explorer operates on the cached Notebook 09 tables. Dropdown changes do not recompute the research pipeline.

Use Mode A to select a canonical strategy. Use Mode B to resolve a historically defined configuration from components. If no exact historical configuration exists, the explorer reports that instead of inventing a new strategy.

In [13]:
cached_leaderboard = canonical_strategy_leaderboard.copy()
cached_configs = configuration_registry.copy()
cached_monthly = monthly_returns_all.copy()
cached_risk = overall_risk_dd_summary.copy()
cached_cost = cost_sensitivity.copy()

def strategy_summary(strategy, benchmark="Baseline carry"):
    row = cached_leaderboard[cached_leaderboard["strategy"].astype(str).eq(str(strategy))].head(1)
    if row.empty:
        return pd.DataFrame({"message": [f"Strategy not found: {strategy}"]})
    r = row.iloc[0]
    out = {
        "Identity": r.get("strategy", ""),
        "Historical status": r.get("historical_research_status", ""),
        "Excluded?": bool(str(r.get("classification_tier", "")).lower().find("excluded") >= 0),
        "Source": r.get("source_notebook", ""),
        "Full return": r.get("full_return", np.nan),
        "Full vol": r.get("full_vol", np.nan),
        "Full Sharpe": r.get("full_sharpe", np.nan),
        "MDD": r.get("full_mdd", np.nan),
        "ES": r.get("full_es", np.nan),
        "Evaluation Sharpe": r.get("evaluation_sharpe", np.nan),
        "Delta eval Sharpe vs Baseline": r.get("evaluation_delta_sharpe_vs_baseline", np.nan),
        "Delta eval Sharpe vs Parent": r.get("evaluation_delta_sharpe_vs_parent", np.nan),
        "Average gross": r.get("average_total_gross", np.nan),
        "Turnover": r.get("average_one_way_turnover", np.nan),
        "5bp net Sharpe": r.get("net_sharpe_5bp", np.nan),
        "Overall systematic RiskDDZ": r.get("overall_systematic_risk_dd_z", np.nan),
        "Fixed crisis RiskDDZ": r.get("overall_fixed_crisis_risk_dd_z", np.nan),
        "Overall RiskDDZ": r.get("overall_risk_dd_z", np.nan),
        "Risk DD hit rate": r.get("risk_dd_hit_rate", np.nan),
        "PIT status": r.get("point_in_time_status", ""),
    }
    return pd.DataFrame(out.items(), columns=["field", "value"])

def resolve_configuration(option_selection="None / Baseline", option_risk_control="None", macro_overlay="None", macro_scale="50%", leg_allocation="None"):
    parts = [option_selection, option_risk_control, macro_overlay, macro_scale, leg_allocation]
    sig = " | ".join(parts).lower()
    candidates = cached_configs.copy()
    mask = pd.Series(True, index=candidates.index)
    if option_selection != "None / Baseline":
        mask &= candidates["display_name"].astype(str).str.lower().str.contains(option_selection.lower(), regex=False)
    else:
        mask &= candidates["display_name"].astype(str).eq("Baseline carry")
    if macro_overlay != "None":
        mask &= candidates["display_name"].astype(str).str.lower().str.contains(macro_overlay.lower(), regex=False)
    hit = candidates[mask].head(1)
    if hit.empty:
        return pd.DataFrame({"message": ["No historically defined configuration matches this selection."], "signature": [sig]})
    return hit[["configuration_id","display_name","canonical_strategy_id","historical_research_status","source_notebook","historically_excluded","exclusion_reason","constructible","construction_status"]]

def diagnostic_plot_pack(strategy, benchmark="Baseline carry"):
    if cached_monthly.empty:
        display(Markdown("Monthly returns are not available for this strategy in the cached monthly source tables."))
        return None
    d = cached_monthly[cached_monthly["strategy"].astype(str).eq(str(strategy))].sort_values("month_end")
    b = cached_monthly[cached_monthly["strategy"].astype(str).eq(str(benchmark))].sort_values("month_end")
    if d.empty:
        display(Markdown(f"No cached monthly return stream for `{strategy}`."))
        return None
    fig, axes = plt.subplots(3, 2, figsize=(13, 11))
    ax = axes[0, 0]
    ax.plot(d["month_end"], (1 + d["gross_return"].fillna(0)).cumprod(), label=strategy)
    if not b.empty:
        ax.plot(b["month_end"], (1 + b["gross_return"].fillna(0)).cumprod(), label=benchmark)
    ax.set_title("NAV vs benchmark")
    ax.legend(fontsize=8)
    ax = axes[0, 1]
    nav = (1 + d["gross_return"].fillna(0)).cumprod()
    ax.plot(d["month_end"], nav / nav.cummax() - 1)
    ax.set_title("Drawdown")
    ax = axes[1, 0]
    for win in [12, 24, 36]:
        roll = d["gross_return"].rolling(win).mean() * 12 / (d["gross_return"].rolling(win).std() * np.sqrt(12))
        ax.plot(d["month_end"], roll, label=f"{win}m")
    ax.set_title("Rolling Sharpe")
    ax.legend()
    ax = axes[1, 1]
    if "gross_roll_notional" in d:
        ax.plot(d["month_end"], d["gross_roll_notional"], label="gross")
    if "net_exposure" in d:
        ax.plot(d["month_end"], d["net_exposure"], label="net")
    ax.set_title("Gross and net exposure")
    ax.legend()
    ax = axes[2, 0]
    if "one_way_turnover" in d:
        ax.plot(d["month_end"], d["one_way_turnover"])
    ax.set_title("One-way turnover")
    ax = axes[2, 1]
    ax.hist(d["gross_return"].dropna(), bins=30, color="#355C7D", alpha=0.8)
    ax.set_title("Monthly return distribution")
    fig.suptitle(f"Diagnostic package: {strategy}", y=1.02)
    fig.tight_layout()
    display(fig)
    plt.close(fig)
    display(strategy_summary(strategy, benchmark))
    return fig

def compare_strategies(strategy_a, strategy_b="Baseline carry"):
    rows = []
    for s in [strategy_a, strategy_b]:
        row = cached_leaderboard[cached_leaderboard["strategy"].astype(str).eq(str(s))].head(1)
        if row.empty:
            continue
        rows.append(row.iloc[0])
    if len(rows) < 2:
        return pd.DataFrame({"message": ["One or both strategies not found."]})
    a, b = rows
    return pd.DataFrame({
        "metric": ["full_return", "full_vol", "full_sharpe", "full_mdd", "full_es", "average_total_gross", "average_one_way_turnover", "evaluation_sharpe", "overall_systematic_risk_dd_z", "overall_fixed_crisis_risk_dd_z", "overall_risk_dd_z"],
        str(strategy_a): [a.get(x, np.nan) for x in ["full_return","full_vol","full_sharpe","full_mdd","full_es","average_total_gross","average_one_way_turnover","evaluation_sharpe","overall_systematic_risk_dd_z","overall_fixed_crisis_risk_dd_z","overall_risk_dd_z"]],
        str(strategy_b): [b.get(x, np.nan) for x in ["full_return","full_vol","full_sharpe","full_mdd","full_es","average_total_gross","average_one_way_turnover","evaluation_sharpe","overall_systematic_risk_dd_z","overall_fixed_crisis_risk_dd_z","overall_risk_dd_z"]],
    }).assign(delta=lambda x: x[str(strategy_a)] - x[str(strategy_b)])

ipywidgets_available = False
manual_fallback_available = True
try:
    import ipywidgets as widgets
    from IPython.display import clear_output
    ipywidgets_available = True
except Exception:
    widgets = None

if ipywidgets_available:
    strategy_options = sorted(cached_leaderboard["strategy"].dropna().astype(str).tolist())
    dd_strategy = widgets.Dropdown(options=strategy_options, value="Baseline carry", description="Strategy")
    dd_benchmark = widgets.Dropdown(options=strategy_options, value="Baseline carry", description="Benchmark")
    output = widgets.Output()
    def _refresh(change=None):
        with output:
            clear_output(wait=True)
            display(strategy_summary(dd_strategy.value, dd_benchmark.value))
            display(compare_strategies(dd_strategy.value, dd_benchmark.value))
    dd_strategy.observe(_refresh, names="value")
    dd_benchmark.observe(_refresh, names="value")
    display(widgets.VBox([widgets.HBox([dd_strategy, dd_benchmark]), output]))
    _refresh()
else:
    EXPLORER_SELECTION = {"strategy": "Baseline carry", "benchmark": "Core A+C"}
    display(Markdown("ipywidgets is unavailable; use EXPLORER_SELECTION and call strategy_summary or diagnostic_plot_pack manually."))
    display(strategy_summary(EXPLORER_SELECTION["strategy"], EXPLORER_SELECTION["benchmark"]))

test_strategies = [
    "Baseline carry", "Core A+B", "Core A+C", "Core A+B+C",
    "OIL_return_1m gross scaling 50%",
    canonical_strategy_leaderboard[canonical_strategy_leaderboard["classification_tier"].astype(str).str.contains("exploratory", case=False, na=False)]["strategy"].head(1).iloc[0],
    "Core A+C + OIL_return_1m",
]
test_rows = []
for s in test_strategies:
    summ = strategy_summary(s)
    test_rows.append({"test_strategy": s, "status": "pass" if "message" not in summ.columns else "fail"})
cmp = compare_strategies("Core A+C", "Core A+B+C")
test_rows.append({"test_strategy": "comparison: Core A+C vs Core A+B+C", "status": "pass" if "message" not in cmp.columns else "fail"})
interactive_test_audit = pd.DataFrame(test_rows)
interactive_test_audit.to_parquet(OUTPUTS["interactive_test_audit"], index=False)
display_df("Interactive explorer test audit", interactive_test_audit, n=20)

**Interactive explorer test audit**

,test_strategy,status
0,Baseline carry,pass
1,Core A+B,pass
2,Core A+C,pass
3,Core A+B+C,pass
4,OIL_return_1m gross scaling 50%,pass
5,A+C x Continuous macro-risk rule,pass
6,Core A+C + OIL_return_1m,pass
7,comparison: Core A+C vs Core A+B+C,pass


## 12. Final Coverage Summary

The final summary records counts, source hashes, top descriptive rankings, risk-period coverage, reconciliation status, and generated output paths.

In [14]:
def top_names(df, col, n=10, ascending=False):
    if df is None or df.empty or col not in df:
        return []
    return df.sort_values(col, ascending=ascending, na_position="last")[["strategy", col]].head(n).to_dict("records")

summary = {
    "source_coverage": source_audit[["source_role","resolved_path","sha256"]].to_dict("records"),
    "historical_configuration_count": int(len(configuration_registry)),
    "constructible_configuration_count": int(configuration_registry["constructible"].sum()),
    "unconstructible_configuration_count": int((~configuration_registry["constructible"]).sum()),
    "canonical_strategy_count": int(len(canonical_strategy_leaderboard)),
    "notebook08_canonical_strategy_count": int(len(reg08)),
    "duplicate_alias_count": int(configuration_registry["is_duplicate_alias"].sum()),
    "historical_primary_count": int(configuration_registry["historically_primary"].sum()),
    "historical_robustness_count": int(configuration_registry["historical_research_status"].astype(str).str.contains("robustness", na=False).sum()),
    "historical_exploratory_count": int(configuration_registry["historically_exploratory"].sum()),
    "historical_sensitivity_count": int(configuration_registry["historically_sensitivity"].sum()),
    "historical_excluded_count": int(configuration_registry["historically_excluded"].sum()),
    "historical_superseded_count": int(configuration_registry["historically_superseded"].sum()),
    "standalone_option_configurations": int(configuration_registry["strategy_family"].astype(str).str.contains("option|S1|S2|S3|S4|S5|S6|S7", case=False, na=False).sum()),
    "risk_scaling_configurations": int(configuration_registry["display_name"].astype(str).str.contains("risk scaling|gross scaling|zero scaling", case=False, na=False).sum()),
    "old_historical_option_filter_configurations": int(configuration_registry["source_notebook"].astype(str).str.contains("06_options_skew", na=False).sum()),
    "abcd_factorial_count": int(len(abcd_factorial_registry)),
    "macro_feature_count": int(len(macro_features)),
    "standalone_macro_overlay_configurations": int(len(macro_overlay_results)),
    "composite_macro_configurations": int(len(macro_composites)),
    "option_macro_configurations": int(len(option_macro_crossproduct)),
    "top_10_full_sample_sharpe": top_names(canonical_strategy_leaderboard, "full_sharpe"),
    "top_10_frozen_evaluation_sharpe": top_names(canonical_strategy_leaderboard, "evaluation_sharpe"),
    "top_10_lowest_mdd": top_names(canonical_strategy_leaderboard, "full_mdd", ascending=False),
    "top_10_best_es": top_names(canonical_strategy_leaderboard, "full_es", ascending=False),
    "top_10_5bp_net_sharpe": top_names(canonical_strategy_leaderboard, "net_sharpe_5bp"),
    "top_10_parent_relative_sharpe": top_names(canonical_strategy_leaderboard, "evaluation_delta_sharpe_vs_parent"),
    "risk_signal_count": int(risk_signal_summary["period_name"].nunique()) if not risk_signal_summary.empty else 0,
    "risk_episode_count": int(risk_episode_detail[["period_name","episode_id"]].drop_duplicates().shape[0]) if not risk_episode_detail.empty else 0,
    "broad_risk_family_count": int(risk_family_summary["risk_family"].nunique()) if not risk_family_summary.empty else 0,
    "fixed_crisis_count": int(fixed_crisis_summary["period_name"].nunique()) if not fixed_crisis_summary.empty else 0,
    "top_10_overall_systematic_risk_dd_z": top_names(canonical_strategy_leaderboard, "overall_systematic_risk_dd_z"),
    "top_10_overall_fixed_crisis_risk_dd_z": top_names(canonical_strategy_leaderboard, "overall_fixed_crisis_risk_dd_z"),
    "top_10_overall_risk_dd_z": top_names(canonical_strategy_leaderboard, "overall_risk_dd_z"),
    "top_10_risk_dd_hit_rate": top_names(canonical_strategy_leaderboard, "risk_dd_hit_rate"),
    "abcd_all_16_present": bool(len(abcd_factorial_registry) == 16 and abcd_factorial_registry["constructible"].all()),
    "abcd_marginal_sharpe_effects": abcd_factorial_effects[abcd_factorial_effects["metric"].eq("full_sharpe")][["module","average_marginal_effect"]].to_dict("records"),
    "notebook07_reconciliation_failures": int(notebook07_reconciliation_audit["status"].eq("fail").sum()) if not notebook07_reconciliation_audit.empty else 0,
    "notebook08_reconciliation_failures": int(notebook08_reconciliation_audit["status"].eq("fail").sum()),
    "duplicate_canonical_ids": int(canonical_strategy_leaderboard["canonical_strategy_id"].duplicated().sum()),
    "undocumented_parameter_values_introduced": 0,
    "risk_dd_aggregation_failures": int(integrity_audit[integrity_audit["check"].astype(str).str.contains("RiskDD|risk episode|systematic", case=False, na=False)]["status"].eq("fail").sum()),
    "pit_future_horizon_violations": int(pit08["future_horizon_leakage"].fillna(False).sum()) if not pit08.empty else None,
    "other_integrity_failures": integrity_audit[integrity_audit["status"].eq("fail")]["check"].tolist(),
    "ipywidgets_available": bool(ipywidgets_available),
    "manual_fallback_available": bool(manual_fallback_available),
    "interactive_tested_strategies": test_strategies,
    "interactive_comparison_mode_status": interactive_test_audit.loc[interactive_test_audit["test_strategy"].str.contains("comparison"), "status"].iloc[0],
    "excel_workbook_path": str(OUTPUTS["excel_workbook"]),
    "parquet_directory": str(OUT_DIR),
    "figure_directory": str(FIG_DIR),
    "notebook_path": str(NOTEBOOK_PATH),
    "figure_paths": figure_paths,
    "outputs": {k: str(v) for k, v in OUTPUTS.items()},
}
OUTPUTS["summary"].write_text(json.dumps(summary, indent=2, default=str))

display(Markdown("### Final Printed Summary\n" + "\n".join([
    "**SOURCE COVERAGE**",
    f"- Exact source paths: {[x['resolved_path'] for x in summary['source_coverage']]}",
    f"- Source hashes saved: {len(summary['source_coverage'])}",
    "",
    "**STRATEGY UNIVERSE**",
    f"- Historical configuration count: {summary['historical_configuration_count']}",
    f"- Constructible / unconstructible: {summary['constructible_configuration_count']} / {summary['unconstructible_configuration_count']}",
    f"- Canonical strategy count: {summary['canonical_strategy_count']}",
    f"- Duplicate alias count: {summary['duplicate_alias_count']}",
    f"- Historical primary / robustness / exploratory / sensitivity / excluded / superseded: {summary['historical_primary_count']} / {summary['historical_robustness_count']} / {summary['historical_exploratory_count']} / {summary['historical_sensitivity_count']} / {summary['historical_excluded_count']} / {summary['historical_superseded_count']}",
    "",
    "**OPTION COVERAGE**",
    f"- Standalone option configurations: {summary['standalone_option_configurations']}",
    f"- Risk-scaling configurations: {summary['risk_scaling_configurations']}",
    f"- Old historical option/filter configurations: {summary['old_historical_option_filter_configurations']}",
    f"- A/B/C/D factorial count: {summary['abcd_factorial_count']}; all 16 present: {summary['abcd_all_16_present']}",
    "",
    "**MACRO COVERAGE**",
    f"- Macro feature count: {summary['macro_feature_count']}",
    f"- Standalone macro overlay configurations: {summary['standalone_macro_overlay_configurations']}",
    f"- Composite macro configurations: {summary['composite_macro_configurations']}",
    f"- Option x macro configurations: {summary['option_macro_configurations']}",
    "",
    "**PERFORMANCE**",
    f"- Top 10 full-sample Sharpe: {[x['strategy'] for x in summary['top_10_full_sample_sharpe']]}",
    f"- Top 10 frozen-evaluation Sharpe: {[x['strategy'] for x in summary['top_10_frozen_evaluation_sharpe']]}",
    f"- Top 10 5bp net Sharpe: {[x['strategy'] for x in summary['top_10_5bp_net_sharpe']]}",
    "",
    "**RISK PERIOD**",
    f"- Risk signal / episode / family / fixed crisis counts: {summary['risk_signal_count']} / {summary['risk_episode_count']} / {summary['broad_risk_family_count']} / {summary['fixed_crisis_count']}",
    f"- Top 10 OverallSystematicRiskDDZ: {[x['strategy'] for x in summary['top_10_overall_systematic_risk_dd_z']]}",
    f"- Top 10 OverallFixedCrisisRiskDDZ: {[x['strategy'] for x in summary['top_10_overall_fixed_crisis_risk_dd_z']]}",
    f"- Top 10 OverallRiskDDZ: {[x['strategy'] for x in summary['top_10_overall_risk_dd_z']]}",
    "",
    "**ABCD**",
    f"- Marginal Sharpe effects: {summary['abcd_marginal_sharpe_effects']}",
    "",
    "**RECONCILIATION AND INTEGRITY**",
    f"- Notebook 07 reconciliation failures: {summary['notebook07_reconciliation_failures']}",
    f"- Notebook 08 reconciliation failures: {summary['notebook08_reconciliation_failures']}",
    f"- Duplicate canonical IDs: {summary['duplicate_canonical_ids']}",
    f"- PIT future-horizon violations: {summary['pit_future_horizon_violations']}",
    f"- Other integrity failures: {summary['other_integrity_failures']}",
    "",
    "**INTERACTIVE EXPLORER**",
    f"- ipywidgets available: {summary['ipywidgets_available']}",
    f"- Manual fallback available: {summary['manual_fallback_available']}",
    f"- Tested strategies: {summary['interactive_tested_strategies']}",
    f"- Comparison-mode test status: {summary['interactive_comparison_mode_status']}",
    "",
    "**OUTPUTS**",
    f"- Excel workbook path: `{summary['excel_workbook_path']}`",
    f"- Parquet directory: `{summary['parquet_directory']}`",
    f"- Figure directory: `{summary['figure_directory']}`",
    f"- Notebook path: `{summary['notebook_path']}`",
])))

### Final Printed Summary
**SOURCE COVERAGE**
- Exact source paths: ['/Users/theoli/Documents/Work/UChicago/Courses/36000_project_lab/BofA/repo/theo/06_options_skew_and_vol_filters.ipynb', '/Users/theoli/Documents/Work/UChicago/Courses/36000_project_lab/BofA/repo/theo/06_options_filter_regression(9).ipynb', '/Users/theoli/Documents/Work/UChicago/Courses/36000_project_lab/BofA/repo/theo/07_option_conditioned_carry_strategy.ipynb', '/Users/theoli/Documents/Work/UChicago/Courses/36000_project_lab/BofA/repo/theo/08_comprehensive_carry_strategy_optimization(6).ipynb']
- Source hashes saved: 4

**STRATEGY UNIVERSE**
- Historical configuration count: 429
- Constructible / unconstructible: 429 / 0
- Canonical strategy count: 371
- Duplicate alias count: 59
- Historical primary / robustness / exploratory / sensitivity / excluded / superseded: 17 / 60 / 266 / 35 / 52 / 44

**OPTION COVERAGE**
- Standalone option configurations: 71
- Risk-scaling configurations: 117
- Old historical option/filter configurations: 44
- A/B/C/D factorial count: 16; all 16 present: True

**MACRO COVERAGE**
- Macro feature count: 25
- Standalone macro overlay configurations: 63
- Composite macro configurations: 6
- Option x macro configurations: 200

**PERFORMANCE**
- Top 10 full-sample Sharpe: ['A+C x Continuous macro-risk rule', 'Core A+B + MXEF_momentum_3m', 'Core A+C + MXEF_momentum_3m', 'Core A+B + SPX_momentum_3m', 'Core A+C + COMMODITY_momentum_3m', 'Core A+C + MOVE_change_1m', 'Core A+C + MXEF_return_1m', 'Core A+C + SPX_momentum_3m', 'Core A+B + OIL_return_1m', 'Core A+B + MXEF_return_1m']
- Top 10 frozen-evaluation Sharpe: ['Baseline carry parent-relative tilt delta=+0.25', 'Baseline carry parent-relative tilt delta=+0.50', 'Core A+C parent-relative tilt delta=+0.25', 'Baseline carry + OIL_return_1m parent-relative tilt delta=+0.25', 'Baseline carry + OIL_return_1m parent-relative tilt delta=+0.50', 'G10-aware ATM conditioning g10=0.5 em=0 + MXEF_momentum_3m', 'G10-aware ATM conditioning g10=0.5 em=0 + JPMVXYEM_level', 'G10-aware ATM conditioning g10=0.5 em=0 + MXEF_drawdown_6m', 'G10-aware ATM conditioning g10=0.5 em=0 + COMMODITY_return_1m', 'Core A+C + MXEF_momentum_3m']
- Top 10 5bp net Sharpe: ['A+C x Continuous macro-risk rule', 'Core A+C + MXEF_momentum_3m', 'Core A+B + SPX_momentum_3m', 'Core A+B + MXEF_momentum_3m', 'Core A+C + COMMODITY_momentum_3m', 'G10-aware ATM conditioning g10=0.5 em=0 + MXEF_return_1m', 'Core A+C + SPX_momentum_3m', 'Core A+C + MXEF_return_1m', 'Core A+C + MOVE_change_1m', 'Core A+B + OIL_return_1m']

**RISK PERIOD**
- Risk signal / episode / family / fixed crisis counts: 28 / 651 / 9 / 5
- Top 10 OverallSystematicRiskDDZ: ['COMMODITY_momentum_3m zero scaling', 'JPMVXYEM_level zero scaling', 'JPMVXYEM_change_1m zero scaling', 'SPX_drawdown_6m zero scaling', 'SPX_return_1m zero scaling', 'DXY_momentum_3m zero scaling', 'SPX_momentum_3m zero scaling', 'MOVE_level zero scaling', 'VIX_change_3m zero scaling', 'OIL_momentum_3m zero scaling']
- Top 10 OverallFixedCrisisRiskDDZ: ['A+C x Confirmed market stress', 'A+C x OR', 'A+C x Two-or-more confirmation', 'B+C x Two-or-more confirmation', 'B+C x OR', 'B+C x Confirmed market stress', 'A+B+C x Two-or-more confirmation', 'A+B+C x OR', 'A+B+C x Confirmed market stress', 'C x Confirmed market stress']
- Top 10 OverallRiskDDZ: ['COMMODITY_momentum_3m zero scaling', 'A+C x OR', 'MXEF_drawdown_6m zero scaling', 'C x OR', 'MOVE_level zero scaling', 'MXEF_momentum_3m zero scaling', 'A+C x Two-or-more confirmation', 'A+C x Confirmed market stress', 'C x Two-or-more confirmation', 'B+C x Continuous macro-risk rule']

**ABCD**
- Marginal Sharpe effects: [{'module': 'A', 'average_marginal_effect': 0.04167839690699239}, {'module': 'B', 'average_marginal_effect': -0.01384778065327373}, {'module': 'C', 'average_marginal_effect': -0.04392121417046298}, {'module': 'D', 'average_marginal_effect': 0.037217739807241204}]

**RECONCILIATION AND INTEGRITY**
- Notebook 07 reconciliation failures: 0
- Notebook 08 reconciliation failures: 0
- Duplicate canonical IDs: 0
- PIT future-horizon violations: 0
- Other integrity failures: []

**INTERACTIVE EXPLORER**
- ipywidgets available: True
- Manual fallback available: True
- Tested strategies: ['Baseline carry', 'Core A+B', 'Core A+C', 'Core A+B+C', 'OIL_return_1m gross scaling 50%', 'A+C x Continuous macro-risk rule', 'Core A+C + OIL_return_1m']
- Comparison-mode test status: pass

**OUTPUTS**
- Excel workbook path: `/Users/theoli/Documents/Work/UChicago/Courses/36000_project_lab/BofA/repo/theo/data/processed/09_strategy_universe/09_strategy_universe_results.xlsx`
- Parquet directory: `/Users/theoli/Documents/Work/UChicago/Courses/36000_project_lab/BofA/repo/theo/data/processed/09_strategy_universe`
- Figure directory: `/Users/theoli/Documents/Work/UChicago/Courses/36000_project_lab/BofA/repo/theo/data/processed/09_strategy_universe/figures`
- Notebook path: `/Users/theoli/Documents/Work/UChicago/Courses/36000_project_lab/BofA/repo/theo/09_comprehensive_strategy_universe_and_interactive_explorer.ipynb`